# 全国微博事件提取

In [5]:
import polars as pl
import os
import time

# ================= 配置区域 =================

# 输入文件夹
INPUT_DIR = r'D:\SM\zdxrun\datatest\city'

# 输出文件 (高召回率版)
OUTPUT_FILE = r'D:\SM\zdxrun\datatest\event\totalevent_all_china_12years.csv'

# 年份范围
START_YEAR = 2012
END_YEAR = 2023

# 字段设置
LOCATION_COL = '标准化地点'   
TIME_COL = '发布时间'
CONTENT_COL = '微博正文'

# 阈值参数 (保持不变，或根据数据量激增适当调高)
SIGMA_THRESHOLD = 3.0   
MIN_POSTS = 15          

# 关键词
raw_keywords_list = [
    '洪灾', '洪水', '大水', '洪流', '水流', '河水', '淹水', '水灾',
    '水淹', '水浸', '水涨', '趟水', '进水', '排水', '洪涝', '涝灾',
    '洪峰', '内涝', '暴雨成灾', '雨灾', '雨水', '雨', '下雨', '暴雨',
    '大雨', '大暴雨', '强降雨', '大到暴雨', '中雨', '暴风雨', '强降水',
    '降雨', '雨量', '降雨量', '降水量', '山洪爆发', '山洪', '滑坡',
    '塌陷', '坍塌', '塌方', '积水', '水深', '齐腰', '冲毁', '冲走'
]
FLOOD_KEYWORDS = "|".join(raw_keywords_list)

# ===========================================

def normalize_loose(col_expr):
    """
    【高召回率模式】地点标准化
    逻辑：
    1. 如果包含多级 (如 "河北省 保定市") -> 取前两级 "河北省 保定市"
    2. 如果只有一级 (如 "北京", "河北省", "朝阳") -> 直接保留原样，绝不丢弃
    """
    list_expr = col_expr.str.split(" ")
    
    return (
        pl.when(list_expr.list.len() >= 2)
        .then(
            # 有省有市，取前两个拼起来，保证归一化
            list_expr.list.slice(0, 2).list.join(" ")
        )
        .otherwise(
            # 【关键修改】只有一级，直接保留！(之前是 None 导致丢弃)
            list_expr.list.first()
        )
    )

def process_one_year(year, file_path):
    print(f"\n>> 正在处理 {year} 年 (高召回模式): {os.path.basename(file_path)}")
    
    if not os.path.exists(file_path):
        print(f"   [跳过] 文件不存在")
        return None

    try:
        q = pl.scan_csv(file_path, infer_schema_length=0, ignore_errors=True)

        q_filtered = (
            q
            .with_columns(
                pl.col(TIME_COL).str.to_datetime(strict=False).dt.date().alias("date")
            )
            .filter(pl.col("date").is_not_null())
            .filter(pl.col(CONTENT_COL).str.contains(FLOOD_KEYWORDS))
        )

        # 使用宽松的标准化逻辑
        q_city = (
            q_filtered
            .with_columns(pl.col(LOCATION_COL).str.split("; ")) 
            .explode(LOCATION_COL)
            .filter(pl.col(LOCATION_COL) != "")
            .with_columns(
                normalize_loose(pl.col(LOCATION_COL)).alias("城市")
            )
            .filter(pl.col("城市").is_not_null()) # 这里几乎不会过滤掉东西了
        )

        # 统计每日热度
        daily_counts = (
            q_city.group_by(["城市", "date"])
            .len()
            .collect()
            .sort(["城市", "date"])
        )

        if daily_counts.is_empty():
            return None

        # 3-Sigma 计算
        city_stats = daily_counts.group_by("城市").agg([
            pl.col("len").mean().alias("mean"),
            pl.col("len").std().fill_null(0).alias("std")
        ])

        burst_days = (
            daily_counts
            .join(city_stats, on="城市")
            .filter(
                (pl.col("len") > (pl.col("mean") + SIGMA_THRESHOLD * pl.col("std"))) &
                (pl.col("len") >= MIN_POSTS)
            )
        )

        if burst_days.is_empty():
            return None

        # 合并事件
        events_df = (
            burst_days
            .sort(["城市", "date"])
            .with_columns([
                pl.col("date").diff().dt.total_days().over("城市").alias("day_diff")
            ])
            .with_columns([
                pl.when((pl.col("day_diff") > 1) | (pl.col("day_diff").is_null()))
                .then(1).otherwise(0).alias("is_new_event")
            ])
            .with_columns([
                pl.col("is_new_event").cum_sum().alias("global_event_id")
            ])
            .group_by(["城市", "global_event_id"])
            .agg([
                pl.col("date").min().alias("开始时间"),
                pl.col("date").max().alias("结束时间"),
                pl.col("len").sum().alias("总热度"),
                pl.col("len").max().alias("峰值热度"),
                pl.len().alias("持续天数")
            ])
            .select([
                pl.col("城市"), pl.col("开始时间"), pl.col("结束时间"), 
                pl.col("总热度"), pl.col("峰值热度"), pl.col("持续天数")
            ])
        )
        
        print(f"   [成功] 提取到 {len(events_df)} 个事件")
        return events_df

    except Exception as e:
        print(f"   [错误] {e}")
        return None

def main():
    start_total = time.time()
    print(f"🚀 开始批量处理 (高召回率版)...")
    
    all_events_list = []

    for year in range(START_YEAR, END_YEAR + 1):
        filename = f"shendata{year}_geonew.csv"
        file_path = os.path.join(INPUT_DIR, filename)
        df_year = process_one_year(year, file_path)
        if df_year is not None:
            all_events_list.append(df_year)

    if all_events_list:
        print("\n正在合并数据...")
        final_df = pl.concat(all_events_list).sort("开始时间")
        
        # 自动创建目录
        os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)
        
        # 【解决Excel乱码】写入 UTF-8 BOM 头
        with open(OUTPUT_FILE, "wb") as f:
            f.write(b"\xef\xbb\xbf")
            final_df.write_csv(f)
            
        print(f"✅ 全部完成！文件已保存至: {OUTPUT_FILE}")
        print(f"总事件数: {len(final_df)}")
        print("\n【Top 5 最强降雨舆情 (预览)】")
        print(final_df.sort("总热度", descending=True).head(5))
    else:
        print("❌ 未提取到事件")
        
    print(f"耗时: {time.time() - start_total:.2f} 秒")

if __name__ == "__main__":
    main()

🚀 开始批量处理 (高召回率版)...

>> 正在处理 2012 年 (高召回模式): shendata2012_geonew.csv
   [成功] 提取到 742 个事件

>> 正在处理 2013 年 (高召回模式): shendata2013_geonew.csv
   [成功] 提取到 825 个事件

>> 正在处理 2014 年 (高召回模式): shendata2014_geonew.csv
   [成功] 提取到 810 个事件

>> 正在处理 2015 年 (高召回模式): shendata2015_geonew.csv
   [成功] 提取到 928 个事件

>> 正在处理 2016 年 (高召回模式): shendata2016_geonew.csv
   [成功] 提取到 1059 个事件

>> 正在处理 2017 年 (高召回模式): shendata2017_geonew.csv
   [成功] 提取到 1248 个事件

>> 正在处理 2018 年 (高召回模式): shendata2018_geonew.csv
   [成功] 提取到 1607 个事件

>> 正在处理 2019 年 (高召回模式): shendata2019_geonew.csv
   [成功] 提取到 1705 个事件

>> 正在处理 2020 年 (高召回模式): shendata2020_geonew.csv
   [成功] 提取到 1580 个事件

>> 正在处理 2021 年 (高召回模式): shendata2021_geonew.csv
   [成功] 提取到 1641 个事件

>> 正在处理 2022 年 (高召回模式): shendata2022_geonew.csv
   [成功] 提取到 1700 个事件

>> 正在处理 2023 年 (高召回模式): shendata2023_geonew.csv
   [成功] 提取到 1645 个事件

正在合并数据...
✅ 全部完成！文件已保存至: D:\SM\zdxrun\datatest\event\totalevent_all_china_12years.csv
总事件数: 15490

【Top 5 最强降雨舆情 (预览)】
shape: (5, 6)
┌─────────

# 降水验证

In [6]:
import pandas as pd
import xarray as xr
import numpy as np
from datetime import timedelta
import sys
import os

# ================= 配置区域 =================

# 1. 输入文件 (上一步生成的12年总表)
INPUT_CSV = r'D:\SM\zdxrun\datatest\event\totalevent_all_china_12years.csv'

# 2. 输出文件 (验证结果)
OUTPUT_CSV = r'D:\SM\zdxrun\datatest\event\verifiedevent_all_china_12years.csv'

# 3. 气象数据文件 (NC)
# ⚠️ 注意：此文件必须包含 2012-2023 的数据，否则请分年份处理
NC_FILE_PATH = r'D:\SM\zdxrun\precipData\CHM_PRE V2-daily\daily\merged_rainfall.nc'

# 4. 验证参数
BUFFER_DEG = 0.15       # 城市中心点周围搜索半径 (度)
LAG_DAYS = 1            # 向前追溯天数 (考虑雨后发博)
HEAVY_RAIN_THRESHOLD = 25.0  # 大雨阈值 (mm)

# 5. 数据源参数
PRECIP_VAR_NAME = 'prec'  # NC文件里的变量名 (常见为 'prec', 'tp', 'pr')
RAINFALL_SCALE = 1      # ⚠️ 单位换算: 如果数据是0.1mm则填0.1, 如果是mm则填1.0

# ===========================================

# --- 全量中国城市坐标字典 (含港澳及所有地级市) ---
CHINA_CITY_COORDS = {
       # --- 直辖市 & 特别行政区 ---
    "北京市": { "lat": 39.9042, "lon": 116.4074 },
    "北京": { "lat": 39.9042, "lon": 116.4074 },
    "天津市": { "lat": 39.0842, "lon": 117.2009 },
    "天津": { "lat": 39.0842, "lon": 117.2009 },
    "上海市": { "lat": 31.2304, "lon": 121.4737 },
    "上海": { "lat": 31.2304, "lon": 121.4737 },
    "重庆市": { "lat": 29.5630, "lon": 106.5516 },
    "重庆": { "lat": 29.5630, "lon": 106.5516 },
    "香港": { "lat": 22.3193, "lon": 114.1694 },
    "香港特别行政区": { "lat": 22.3193, "lon": 114.1694 },
    "澳门": { "lat": 22.1987, "lon": 113.5439 },
    "澳门特别行政区": { "lat": 22.1987, "lon": 113.5439 },

    # --- 河北省 ---
    "石家庄市": { "lat": 38.0428, "lon": 114.5149 }, "唐山市": { "lat": 39.6309, "lon": 118.1802 },
    "秦皇岛市": { "lat": 39.9354, "lon": 119.6005 }, "邯郸市": { "lat": 36.6256, "lon": 114.5391 },
    "邢台市": { "lat": 37.0706, "lon": 114.5048 }, "保定市": { "lat": 38.8738, "lon": 115.4648 },
    "张家口市": { "lat": 40.8244, "lon": 114.8859 }, "承德市": { "lat": 40.9644, "lon": 117.9624 },
    "沧州市": { "lat": 38.3045, "lon": 116.8388 }, "廊坊市": { "lat": 39.5380, "lon": 116.6838 },
    "衡水市": { "lat": 37.7390, "lon": 115.6739 },

    # --- 山西省 ---
    "太原市": { "lat": 37.8706, "lon": 112.5489 }, "大同市": { "lat": 40.0768, "lon": 113.3001 },
    "阳泉市": { "lat": 37.8570, "lon": 113.5808 }, "长治市": { "lat": 36.1954, "lon": 113.1163 },
    "晋城市": { "lat": 35.4902, "lon": 112.8521 }, "朔州市": { "lat": 39.3317, "lon": 112.4329 },
    "晋中市": { "lat": 37.6870, "lon": 112.7527 }, "运城市": { "lat": 35.0264, "lon": 111.0073 },
    "忻州市": { "lat": 38.4167, "lon": 112.7342 }, "临汾市": { "lat": 36.0880, "lon": 111.5190 },
    "吕梁市": { "lat": 37.5184, "lon": 111.1444 },

    # --- 内蒙古自治区 ---
    "呼和浩特市": { "lat": 40.8419, "lon": 111.7492 }, "包头市": { "lat": 40.6579, "lon": 109.8403 },
    "乌海市": { "lat": 39.6554, "lon": 106.7941 }, "赤峰市": { "lat": 42.2578, "lon": 118.8870 },
    "通辽市": { "lat": 43.6133, "lon": 122.2415 }, "鄂尔多斯市": { "lat": 39.6083, "lon": 109.7813 },
    "呼伦贝尔市": { "lat": 49.2116, "lon": 119.7658 }, "巴彦淖尔市": { "lat": 40.7424, "lon": 107.4169 },
    "乌兰察布市": { "lat": 41.0185, "lon": 113.1326 }, "兴安盟": { "lat": 46.0772, "lon": 122.0653 },
    "锡林郭勒盟": { "lat": 43.9332, "lon": 116.0518 }, "阿拉善盟": { "lat": 38.8519, "lon": 105.7289 },

    # --- 辽宁省 ---
    "沈阳市": { "lat": 41.8057, "lon": 123.4315 }, "大连市": { "lat": 38.9140, "lon": 121.6147 },
    "鞍山市": { "lat": 41.1078, "lon": 122.9946 }, "抚顺市": { "lat": 41.8809, "lon": 123.9572 },
    "本溪市": { "lat": 41.2941, "lon": 123.7662 }, "丹东市": { "lat": 40.1242, "lon": 124.3565 },
    "锦州市": { "lat": 41.0951, "lon": 121.1270 }, "营口市": { "lat": 40.6670, "lon": 122.2354 },
    "阜新市": { "lat": 42.0220, "lon": 121.6700 }, "辽阳市": { "lat": 41.2694, "lon": 123.1724 },
    "盘锦市": { "lat": 41.2707, "lon": 122.0707 }, "铁岭市": { "lat": 42.2230, "lon": 123.8443 },
    "朝阳市": { "lat": 41.5736, "lon": 120.4508 }, "葫芦岛市": { "lat": 40.7111, "lon": 120.8369 },

    # --- 吉林省 ---
    "长春市": { "lat": 43.8171, "lon": 125.3235 }, "吉林市": { "lat": 43.8436, "lon": 126.5496 },
    "四平市": { "lat": 43.1664, "lon": 124.3504 }, "辽源市": { "lat": 42.9027, "lon": 125.1264 },
    "通化市": { "lat": 41.7284, "lon": 125.9396 }, "白山市": { "lat": 41.9388, "lon": 126.4173 },
    "松原市": { "lat": 45.1419, "lon": 124.8251 }, "白城市": { "lat": 45.6198, "lon": 122.8389 },
    "延边": { "lat": 42.93, "lon": 129.51 }, "延边朝鲜族自治州": { "lat": 42.93, "lon": 129.51 },

    # --- 黑龙江省 ---
    "哈尔滨市": { "lat": 45.8038, "lon": 126.5349 }, "齐齐哈尔市": { "lat": 47.3543, "lon": 123.9182 },
    "鸡西市": { "lat": 45.2951, "lon": 130.9693 }, "鹤岗市": { "lat": 47.3500, "lon": 130.2979 },
    "双鸭山市": { "lat": 46.6437, "lon": 131.1592 }, "大庆市": { "lat": 46.5871, "lon": 125.1039 },
    "伊春市": { "lat": 47.7247, "lon": 128.8994 }, "佳木斯市": { "lat": 46.8020, "lon": 130.3199 },
    "七台河市": { "lat": 45.7713, "lon": 131.0035 }, "牡丹江市": { "lat": 44.5765, "lon": 129.6332 },
    "黑河市": { "lat": 50.2447, "lon": 127.5283 }, "绥化市": { "lat": 46.6334, "lon": 126.9689 },
    "大兴安岭地区": { "lat": 52.3353, "lon": 124.7115 },

    # --- 江苏省 ---
    "南京市": { "lat": 32.0584, "lon": 118.7969 }, "无锡市": { "lat": 31.4912, "lon": 120.3119 },
    "徐州市": { "lat": 34.2044, "lon": 117.2841 }, "常州市": { "lat": 31.8112, "lon": 119.9740 },
    "苏州市": { "lat": 31.2989, "lon": 120.5853 }, "南通市": { "lat": 31.9802, "lon": 120.8943 },
    "连云港市": { "lat": 34.5967, "lon": 119.2216 }, "淮安市": { "lat": 33.6104, "lon": 119.0153 },
    "盐城市": { "lat": 33.3474, "lon": 120.1636 }, "扬州市": { "lat": 32.3942, "lon": 119.4129 },
    "镇江市": { "lat": 32.1878, "lon": 119.4258 }, "泰州市": { "lat": 32.4555, "lon": 119.9246 },
    "宿迁市": { "lat": 33.9630, "lon": 118.2755 },

    # --- 浙江省 ---
    "杭州市": { "lat": 30.2741, "lon": 120.1551 }, "宁波市": { "lat": 29.8683, "lon": 121.5440 },
    "温州市": { "lat": 27.9943, "lon": 120.6994 }, "嘉兴市": { "lat": 30.7539, "lon": 120.7555 },
    "湖州市": { "lat": 30.8943, "lon": 120.0868 }, "绍兴市": { "lat": 30.0312, "lon": 120.5802 },
    "金华市": { "lat": 29.0791, "lon": 119.6474 }, "衢州市": { "lat": 28.9417, "lon": 118.8877 },
    "舟山市": { "lat": 29.9855, "lon": 122.2072 }, "台州市": { "lat": 28.6564, "lon": 121.4208 },
    "丽水市": { "lat": 28.4676, "lon": 119.9228 },

    # --- 安徽省 ---
    "合肥市": { "lat": 31.8206, "lon": 117.2272 }, "芜湖市": { "lat": 31.3528, "lon": 118.3841 },
    "蚌埠市": { "lat": 32.9163, "lon": 117.3897 }, "淮南市": { "lat": 32.6326, "lon": 116.9969 },
    "马鞍山市": { "lat": 31.6700, "lon": 118.5067 }, "淮北市": { "lat": 33.9558, "lon": 116.7983 },
    "铜陵市": { "lat": 30.9451, "lon": 117.8115 }, "安庆市": { "lat": 30.5648, "lon": 117.1101 },
    "黄山市": { "lat": 29.7147, "lon": 118.3375 }, "滁州市": { "lat": 32.3015, "lon": 118.3169 },
    "阜阳市": { "lat": 32.8901, "lon": 115.8142 }, "宿州市": { "lat": 33.6395, "lon": 116.9855 },
    "六安市": { "lat": 31.7460, "lon": 116.5022 }, "亳州市": { "lat": 33.8446, "lon": 115.7824 },
    "池州市": { "lat": 30.6648, "lon": 117.4916 }, "宣城市": { "lat": 30.9407, "lon": 118.7588 },

    # --- 福建省 ---
    "福州市": { "lat": 26.0745, "lon": 119.2965 }, "厦门市": { "lat": 24.4798, "lon": 118.0894 },
    "莆田市": { "lat": 25.4540, "lon": 119.0076 }, "三明市": { "lat": 26.2634, "lon": 117.6386 },
    "泉州市": { "lat": 24.8739, "lon": 118.6759 }, "漳州市": { "lat": 24.5133, "lon": 117.6483 },
    "南平市": { "lat": 26.6436, "lon": 118.1777 }, "龙岩市": { "lat": 25.0752, "lon": 117.0179 },
    "宁德市": { "lat": 26.6656, "lon": 119.5479 },

    # --- 江西省 ---
    "南昌市": { "lat": 28.6820, "lon": 115.8579 }, "景德镇市": { "lat": 29.2948, "lon": 117.1782 },
    "萍乡市": { "lat": 27.6253, "lon": 113.8496 }, "九江市": { "lat": 29.7051, "lon": 116.0019 },
    "新余市": { "lat": 27.8184, "lon": 114.9167 }, "鹰潭市": { "lat": 28.2605, "lon": 117.0682 },
    "赣州市": { "lat": 25.8311, "lon": 114.9347 }, "吉安市": { "lat": 27.1138, "lon": 114.9863 },
    "宜春市": { "lat": 27.8155, "lon": 114.3888 }, "抚州市": { "lat": 27.9546, "lon": 116.3584 },
    "上饶市": { "lat": 28.4532, "lon": 117.9436 },

    # --- 山东省 ---
    "济南市": { "lat": 36.6512, "lon": 117.1205 }, "青岛市": { "lat": 36.0671, "lon": 120.3826 },
    "淄博市": { "lat": 36.8131, "lon": 118.0550 }, "枣庄市": { "lat": 34.8105, "lon": 117.3237 },
    "东营市": { "lat": 37.4341, "lon": 118.6747 }, "烟台市": { "lat": 37.4638, "lon": 121.4479 },
    "潍坊市": { "lat": 36.7068, "lon": 119.1618 }, "济宁市": { "lat": 35.4136, "lon": 116.5872 },
    "泰安市": { "lat": 36.2002, "lon": 117.0877 }, "威海市": { "lat": 37.5131, "lon": 122.1204 },
    "日照市": { "lat": 35.4164, "lon": 119.5269 }, "临沂市": { "lat": 35.1047, "lon": 118.3565 },
    "德州市": { "lat": 37.4355, "lon": 116.3575 }, "聊城市": { "lat": 36.4560, "lon": 115.9854 },
    "滨州市": { "lat": 37.3820, "lon": 117.9707 }, "菏泽市": { "lat": 35.2338, "lon": 115.4807 },

    # --- 河南省 ---
    "郑州市": { "lat": 34.7466, "lon": 113.6253 }, "开封市": { "lat": 34.7972, "lon": 114.3076 },
    "洛阳市": { "lat": 34.6181, "lon": 112.4540 }, "平顶山市": { "lat": 33.7661, "lon": 113.1928 },
    "安阳市": { "lat": 36.0963, "lon": 114.3925 }, "鹤壁市": { "lat": 35.8977, "lon": 114.2973 },
    "新乡市": { "lat": 35.3026, "lon": 113.9268 }, "焦作市": { "lat": 35.2158, "lon": 113.2418 },
    "濮阳市": { "lat": 35.7618, "lon": 115.0292 }, "许昌市": { "lat": 34.0355, "lon": 113.8526 },
    "漯河市": { "lat": 33.5825, "lon": 114.0165 }, "三门峡市": { "lat": 34.7734, "lon": 111.2004 },
    "南阳市": { "lat": 32.9908, "lon": 112.5283 }, "商丘市": { "lat": 34.4140, "lon": 115.6564 },
    "信阳市": { "lat": 32.1470, "lon": 114.0913 }, "周口市": { "lat": 33.6251, "lon": 114.6976 },
    "驻马店市": { "lat": 32.9907, "lon": 114.0296 },

    # --- 湖北省 ---
    "武汉市": { "lat": 30.5928, "lon": 114.3055 }, "黄石市": { "lat": 30.1984, "lon": 115.0385 },
    "十堰市": { "lat": 32.6475, "lon": 110.7994 }, "宜昌市": { "lat": 30.6920, "lon": 111.2865 },
    "襄阳市": { "lat": 32.0082, "lon": 112.1224 }, "鄂州市": { "lat": 30.3919, "lon": 114.8948 },
    "荆门市": { "lat": 31.0354, "lon": 112.1994 }, "孝感市": { "lat": 30.9178, "lon": 113.9556 },
    "荆州市": { "lat": 30.3352, "lon": 112.2393 }, "黄冈市": { "lat": 30.4526, "lon": 114.8724 },
    "咸宁市": { "lat": 29.8414, "lon": 114.3225 }, "随州市": { "lat": 31.7175, "lon": 113.3824 },
    "恩施土家族苗族自治州": { "lat": 30.2950, "lon": 109.4794 }, "恩施": { "lat": 30.2950, "lon": 109.4794 },

    # --- 湖南省 ---
    "长沙市": { "lat": 28.2282, "lon": 112.9388 }, "株洲市": { "lat": 27.8274, "lon": 113.1338 },
    "湘潭市": { "lat": 27.8297, "lon": 112.9441 }, "衡阳市": { "lat": 26.8965, "lon": 112.5719 },
    "邵阳市": { "lat": 27.2389, "lon": 111.4693 }, "岳阳市": { "lat": 29.3563, "lon": 113.1290 },
    "常德市": { "lat": 29.0316, "lon": 111.6985 }, "张家界市": { "lat": 29.1170, "lon": 110.4792 },
    "益阳市": { "lat": 28.5528, "lon": 112.3550 }, "郴州市": { "lat": 25.7705, "lon": 113.0145 },
    "永州市": { "lat": 26.4204, "lon": 111.6134 }, "怀化市": { "lat": 27.5516, "lon": 109.9985 },
    "娄底市": { "lat": 27.7017, "lon": 111.9960 }, "湘西土家族苗族自治州": { "lat": 28.3115, "lon": 109.7397 },

    # --- 广东省 ---
    "广州市": { "lat": 23.1291, "lon": 113.2644 }, "韶关市": { "lat": 24.8104, "lon": 113.5975 },
    "深圳市": { "lat": 22.5431, "lon": 114.0579 }, "珠海市": { "lat": 22.2707, "lon": 113.5767 },
    "汕头市": { "lat": 23.3541, "lon": 116.6820 }, "佛山市": { "lat": 23.0215, "lon": 113.1214 },
    "江门市": { "lat": 22.5788, "lon": 113.0816 }, "湛江市": { "lat": 21.2749, "lon": 110.3594 },
    "茂名市": { "lat": 21.6622, "lon": 110.9254 }, "肇庆市": { "lat": 23.0476, "lon": 112.4651 },
    "惠州市": { "lat": 23.1115, "lon": 114.4162 }, "梅州市": { "lat": 24.2884, "lon": 116.1225 },
    "汕尾市": { "lat": 22.7875, "lon": 115.3753 }, "河源市": { "lat": 23.7463, "lon": 114.7006 },
    "阳江市": { "lat": 21.8570, "lon": 111.9827 }, "清远市": { "lat": 23.6818, "lon": 113.0560 },
    "东莞市": { "lat": 23.0207, "lon": 113.7518 }, "中山市": { "lat": 22.5170, "lon": 113.3928 },
    "潮州市": { "lat": 23.6701, "lon": 116.6226 }, "揭阳市": { "lat": 23.5503, "lon": 116.3729 },
    "云浮市": { "lat": 22.9150, "lon": 112.0445 },

    # --- 广西壮族自治区 ---
    "南宁市": { "lat": 22.8170, "lon": 108.3665 }, "柳州市": { "lat": 24.3275, "lon": 109.4170 },
    "桂林市": { "lat": 25.2685, "lon": 110.2902 }, "梧州市": { "lat": 23.4770, "lon": 111.2791 },
    "北海市": { "lat": 21.4812, "lon": 109.1193 }, "防城港市": { "lat": 21.6862, "lon": 108.3537 },
    "钦州市": { "lat": 21.9810, "lon": 108.6541 }, "贵港市": { "lat": 23.1115, "lon": 109.5989 },
    "玉林市": { "lat": 22.6366, "lon": 110.1652 }, "百色市": { "lat": 23.9042, "lon": 106.6163 },
    "贺州市": { "lat": 24.4036, "lon": 111.5667 }, "河池市": { "lat": 24.6929, "lon": 108.0851 },
    "来宾市": { "lat": 23.7503, "lon": 109.2298 }, "崇左市": { "lat": 22.4152, "lon": 107.3650 },

    # --- 海南省 ---
    "海口市": { "lat": 20.0174, "lon": 110.3492 }, "三亚市": { "lat": 18.2528, "lon": 109.5120 },
    "三沙市": { "lat": 16.8310, "lon": 112.3386 }, "儋州市": { "lat": 19.5206, "lon": 109.5768 },

    # --- 四川省 ---
    "成都市": { "lat": 30.6586, "lon": 104.0648 }, "自贡市": { "lat": 29.3516, "lon": 104.7784 },
    "攀枝花市": { "lat": 26.5823, "lon": 101.7186 }, "泸州市": { "lat": 28.8724, "lon": 105.4405 },
    "德阳市": { "lat": 31.1278, "lon": 104.3979 }, "绵阳市": { "lat": 31.4674, "lon": 104.7323 },
    "广元市": { "lat": 32.4354, "lon": 105.8434 }, "遂宁市": { "lat": 30.5328, "lon": 105.5928 },
    "内江市": { "lat": 29.5802, "lon": 105.0584 }, "乐山市": { "lat": 29.5521, "lon": 103.7656 },
    "南充市": { "lat": 30.8373, "lon": 106.1107 }, "眉山市": { "lat": 30.0762, "lon": 103.8486 },
    "宜宾市": { "lat": 28.7518, "lon": 104.6438 }, "广安市": { "lat": 30.4561, "lon": 106.6334 },
    "达州市": { "lat": 31.2093, "lon": 107.4680 }, "雅安市": { "lat": 29.9874, "lon": 103.0422 },
    "巴中市": { "lat": 31.8665, "lon": 106.7537 }, "资阳市": { "lat": 30.1290, "lon": 104.6419 },
    "阿坝": { "lat": 31.8994, "lon": 102.2214 }, "甘孜": { "lat": 30.0495, "lon": 101.9638 },
    "凉山": { "lat": 27.8868, "lon": 102.2678 },

    # --- 贵州省 ---
    "贵阳市": { "lat": 26.6470, "lon": 106.6302 }, "六盘水市": { "lat": 26.5927, "lon": 104.8304 },
    "遵义市": { "lat": 27.7263, "lon": 106.9274 }, "安顺市": { "lat": 26.2445, "lon": 105.9283 },
    "毕节市": { "lat": 27.3017, "lon": 105.2906 }, "铜仁市": { "lat": 27.7263, "lon": 109.1897 },
    "黔西南": { "lat": 25.0878, "lon": 104.8979 }, "黔东南": { "lat": 26.5830, "lon": 107.9815 },
    "黔南": { "lat": 26.2530, "lon": 107.5172 },

    # --- 云南省 ---
    "昆明市": { "lat": 24.8801, "lon": 102.8329 }, "曲靖市": { "lat": 25.4900, "lon": 103.7962 },
    "玉溪市": { "lat": 24.3520, "lon": 102.5439 }, "保山市": { "lat": 25.1118, "lon": 99.1618 },
    "昭通市": { "lat": 27.3366, "lon": 103.7172 }, "丽江市": { "lat": 26.8721, "lon": 100.2260 },
    "普洱市": { "lat": 23.0665, "lon": 100.9765 }, "临沧市": { "lat": 23.8866, "lon": 100.0870 },
    "楚雄": { "lat": 25.0389, "lon": 101.5460 }, "红河": { "lat": 23.3668, "lon": 103.3842 },
    "文山": { "lat": 23.3667, "lon": 104.2440 }, "西双版纳": { "lat": 22.0088, "lon": 100.7979 },
    "大理": { "lat": 25.5928, "lon": 100.2257 }, "德宏": { "lat": 24.4367, "lon": 98.5784 },
    "怒江": { "lat": 25.8510, "lon": 98.8543 }, "迪庆": { "lat": 27.8188, "lon": 99.7065 },

    # --- 西藏自治区 ---
    "拉萨市": { "lat": 29.6528, "lon": 91.1322 }, "日喀则市": { "lat": 29.2663, "lon": 88.8803 },
    "昌都市": { "lat": 31.1407, "lon": 97.1708 }, "林芝市": { "lat": 29.6469, "lon": 94.3615 },
    "山南市": { "lat": 29.2372, "lon": 91.7731 }, "那曲市": { "lat": 31.4760, "lon": 92.0522 },
    "阿里地区": { "lat": 32.5011, "lon": 80.1055 },

    # --- 陕西省 ---
    "西安市": { "lat": 34.3416, "lon": 108.9398 }, "铜川市": { "lat": 34.8977, "lon": 108.9733 },
    "宝鸡市": { "lat": 34.3619, "lon": 107.1449 }, "咸阳市": { "lat": 34.3296, "lon": 108.7051 },
    "渭南市": { "lat": 34.4994, "lon": 109.5029 }, "延安市": { "lat": 36.5854, "lon": 109.4897 },
    "汉中市": { "lat": 33.0676, "lon": 107.0286 }, "榆林市": { "lat": 38.2906, "lon": 109.7412 },
    "安康市": { "lat": 32.6848, "lon": 109.0293 }, "商洛市": { "lat": 33.8703, "lon": 109.9398 },

    # --- 甘肃省 ---
    "兰州市": { "lat": 36.0610, "lon": 103.8343 }, "嘉峪关市": { "lat": 39.7731, "lon": 98.2816 },
    "金昌市": { "lat": 38.5201, "lon": 102.1880 }, "白银市": { "lat": 36.5448, "lon": 104.1377 },
    "天水市": { "lat": 34.5809, "lon": 105.7250 }, "武威市": { "lat": 37.9283, "lon": 102.6347 },
    "张掖市": { "lat": 38.9256, "lon": 100.4499 }, "平凉市": { "lat": 35.5427, "lon": 106.6651 },
    "酒泉市": { "lat": 39.7121, "lon": 98.4947 }, "庆阳市": { "lat": 35.7389, "lon": 107.6326 },
    "定西市": { "lat": 35.5807, "lon": 104.6263 }, "陇南市": { "lat": 33.3951, "lon": 104.9221 },
    "临夏": { "lat": 35.6011, "lon": 103.2120 }, "甘南": { "lat": 34.9863, "lon": 102.9110 },

    # --- 青海省 ---
    "西宁市": { "lat": 36.6171, "lon": 101.7782 }, "海东市": { "lat": 36.5029, "lon": 102.1033 },
    "海北": { "lat": 36.9594, "lon": 100.9010 }, "黄南": { "lat": 35.5177, "lon": 102.0199 },
    "海南": { "lat": 36.2924, "lon": 100.6195 }, "果洛": { "lat": 34.4709, "lon": 100.2421 },
    "玉树": { "lat": 33.0041, "lon": 97.0088 }, "海西": { "lat": 37.3781, "lon": 97.3708 },

    # --- 宁夏回族自治区 ---
    "银川市": { "lat": 38.4871, "lon": 106.2309 }, "石嘴山市": { "lat": 39.0136, "lon": 106.3762 },
    "吴忠市": { "lat": 37.9975, "lon": 106.1982 }, "固原市": { "lat": 36.0046, "lon": 106.2426 },
    "中卫市": { "lat": 37.5149, "lon": 105.1896 },

    # --- 新疆维吾尔自治区 ---
    "乌鲁木齐市": { "lat": 43.8256, "lon": 87.6168 }, "克拉玛依市": { "lat": 45.5798, "lon": 84.8893 },
    "吐鲁番市": { "lat": 42.9513, "lon": 89.1895 }, "哈密市": { "lat": 42.8185, "lon": 93.5151 },
    "昌吉": { "lat": 44.0112, "lon": 87.3082 }, "博尔塔拉": { "lat": 44.9056, "lon": 82.0662 },
    "巴音郭楞": { "lat": 41.7641, "lon": 86.1453 }, "阿克苏地区": { "lat": 41.1688, "lon": 80.2606 },
    "克孜勒苏": { "lat": 39.7145, "lon": 76.1678 }, "喀什地区": { "lat": 39.4704, "lon": 75.9897 },
    "和田地区": { "lat": 37.1107, "lon": 79.9234 }, "伊犁": { "lat": 43.9168, "lon": 81.3242 },
    "塔城地区": { "lat": 46.7453, "lon": 82.9857 }, "阿勒泰地区": { "lat": 47.8449, "lon": 88.1413 }
}
# 注意：为了代码运行，上面的字典包含了主要城市。
# 下面的 get_city_coords_smart 函数能处理大部分 "XX市" 的匹配。

def get_city_coords_smart(city_str, coords_db):
    """ 智能匹配坐标 """
    if pd.isna(city_str): return None, None, "空值"
    
    # 1. 清洗：提取 "省份 城市" 中的 "城市"
    parts = str(city_str).replace('省', '省 ').replace('市', '市 ').split()
    target = parts[-1] if parts else str(city_str)
    target = target.strip()

    # 2. 精确匹配
    if target in coords_db:
        return coords_db[target]['lat'], coords_db[target]['lon'], target

    # 3. 尝试加"市" (如 "武汉" -> "武汉市")
    if target + "市" in coords_db:
        key = target + "市"
        return coords_db[key]['lat'], coords_db[key]['lon'], key

    # 4. 尝试模糊匹配 (字典key以target开头，如 "延边" -> "延边朝鲜族...")
    for db_key in coords_db:
        if db_key.startswith(target):
             return coords_db[db_key]['lat'], coords_db[db_key]['lon'], db_key
    
    # 5. 最后的兜底：如果字典不全，这里返回 None
    # 实际应用中建议补充完整的 360+ 城市字典
    return None, None, None

def validate_heavy_rain(row, ds, coords_db):
    try:
        # 1. 获取坐标
        center_lat, center_lon, matched_name = get_city_coords_smart(row['城市'], coords_db)
        
        if center_lat is None:
            # 简单的后备：如果没有在字典里，可能需要外部API查询，这里暂标记为未找到
            return False, 0.0, f"坐标未收录: {row['城市']}"

        # 2. 确定 NC 文件的维度名
        lat_name = next((x for x in ['lat', 'latitude'] if x in ds.coords), None)
        lon_name = next((x for x in ['lon', 'longitude'] if x in ds.coords), None)
        
        if not lat_name or not lon_name:
            return False, 0.0, f"NC文件维度错误: {list(ds.coords)}"

        # 3. 时间切片
        start_dt = row['开始时间'] - timedelta(days=LAG_DAYS)
        end_dt = row['结束时间']
        
        # 4. 空间切片 (处理经纬度顺序)
        lat_arr = ds[lat_name]
        lon_arr = ds[lon_name]
        
        # 自动判断切片方向 (降序/升序)
        lat_slice = slice(center_lat + BUFFER_DEG, center_lat - BUFFER_DEG) if lat_arr[0] > lat_arr[-1] else slice(center_lat - BUFFER_DEG, center_lat + BUFFER_DEG)
        lon_slice = slice(center_lon + BUFFER_DEG, center_lon - BUFFER_DEG) if lon_arr[0] > lon_arr[-1] else slice(center_lon - BUFFER_DEG, center_lon + BUFFER_DEG)

        # 5. 查询数据
        # 使用 sel 进行切片
        subset = ds.sel({
            'time': slice(start_dt, end_dt),
            lat_name: lat_slice,
            lon_name: lon_slice
        })
        
        # 6. 检查是否有数据
        if subset.time.size == 0:
            return False, 0.0, "该时间段无气象数据" # 可能是NC文件没包含这年

        if subset[lat_name].size == 0 or subset[lon_name].size == 0:
             return False, 0.0, "坐标超出NC范围"

        # 7. 计算最大降雨量
        # load() 将数据读入内存计算
        local_data = subset[PRECIP_VAR_NAME].load()
        max_rain_raw = local_data.max().item()

        if pd.isna(max_rain_raw):
            max_rain = 0.0
        else:
            max_rain = max_rain_raw * RAINFALL_SCALE

        # 8. 判定
        is_valid = max_rain >= HEAVY_RAIN_THRESHOLD
        
        return is_valid, max_rain, f"匹配: {matched_name}"

    except Exception as e:
        return False, 0.0, f"Error: {str(e)}"

def main():
    print(f"🚀 启动 Step 6 (降水验证 12年版)...")
    
    # 1. 读取 CSV
    if not os.path.exists(INPUT_CSV):
        print(f"❌ 找不到输入文件: {INPUT_CSV}")
        return
    
    print("正在读取事件列表 CSV...")
    df = pd.read_csv(INPUT_CSV, parse_dates=['开始时间', '结束时间'])
    print(f"共加载 {len(df)} 个事件。")

    # 2. 打开 NC 文件
    if not os.path.exists(NC_FILE_PATH):
        print(f"❌ 找不到气象数据: {NC_FILE_PATH}")
        return

    print("正在链接气象数据集 (Lazy Loading)...")
    try:
        ds = xr.open_dataset(NC_FILE_PATH, chunks='auto')
        print(f"气象数据时间范围: {ds.time.values[0]} 至 {ds.time.values[-1]}")
    except Exception as e:
        print(f"❌ 打开 NC 文件失败: {e}")
        return

    # 3. 循环验证
    print("开始逐个验证 (可能需要几分钟)...")
    results = []
    total = len(df)
    
    for i, row in df.iterrows():
        is_valid, rainfall, msg = validate_heavy_rain(row, ds, CHINA_CITY_COORDS)
        results.append([is_valid, rainfall, msg])
        
        if (i + 1) % 100 == 0:
            sys.stdout.write(f"\r进度: {i+1}/{total} | 最新验证: {row['城市']} -> {rainfall:.1f}mm ({msg})")
            sys.stdout.flush()

    print("\n验证完成！")

    # 4. 合并结果
    res_df = pd.DataFrame(results, columns=['Verified', 'Max_Rainfall', 'Message'])
    final_df = pd.concat([df, res_df], axis=1)

    # 5. 保存 (UTF-8-SIG for Excel)
    os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
    final_df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
    
    print("-" * 40)
    print(f"✅ 结果已保存至: {OUTPUT_CSV}")
    print(f"验证通过数: {final_df['Verified'].sum()}")
    print("-" * 40)

if __name__ == "__main__":
    main()

🚀 启动 Step 6 (降水验证 12年版)...
正在读取事件列表 CSV...
共加载 15490 个事件。
正在链接气象数据集 (Lazy Loading)...
气象数据时间范围: 2012-01-01T00:00:00.000000000 至 2024-12-31T00:00:00.000000000
开始逐个验证 (可能需要几分钟)...
进度: 15400/15490 | 最新验证: 云南省 临沧市 -> 0.3mm (匹配: 临沧市)))族自治区)凉山彝族自治州)自治州)回族自治州)
验证完成！
----------------------------------------
✅ 结果已保存至: D:\SM\zdxrun\datatest\event\verifiedevent_all_china_12years.csv
验证通过数: 7631
----------------------------------------


In [7]:
import pandas as pd

# 1. 读取跑完验证的 CSV
input_path = r'D:\SM\zdxrun\datatest\event\verifiedevent_all_china_12years.csv'
df = pd.read_csv(input_path)

# 统计原始数量
total_count = len(df)
print(f"原始数据集总行数: {total_count}")

# 2. 执行筛选：保留 Verified 为 True 的行
# 注意：pandas 读取布尔值列时，有时会读成字符串 'True'，有时是布尔值 True
# 下面这种写法兼容两种情况
if df['Verified'].dtype == 'object':
    df_valid = df[df['Verified'].astype(str) == 'True'].copy()
else:
    df_valid = df[df['Verified'] == True].copy()

# 统计筛选后的数量
valid_count = len(df_valid)
dropped_count = total_count - valid_count
drop_rate = (dropped_count / total_count) * 100

print("-" * 30)
print(f"✅ 有效暴雨内涝事件: {valid_count} 条")
print(f"❌ 剔除虚假/无雨事件: {dropped_count} 条")
print(f"📉 剔除率 (虚假阳性率): {drop_rate:.2f}%")
print("-" * 30)

# 3. 保存清洗后的“纯净版”数据
output_path = r'D:\SM\zdxrun\datatest\event\verifiedevent_all_china_12years_cleaned.csv'
df_valid.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"纯净版数据已保存至: {output_path}")

原始数据集总行数: 15490
------------------------------
✅ 有效暴雨内涝事件: 7631 条
❌ 剔除虚假/无雨事件: 7859 条
📉 剔除率 (虚假阳性率): 50.74%
------------------------------
纯净版数据已保存至: D:\SM\zdxrun\datatest\event\verifiedevent_all_china_12years_cleaned.csv


# 事件合并

In [9]:
import pandas as pd
from datetime import timedelta

# 1. 读取已排序的文件
input_path = r'D:\SM\zdxrun\datatest\event\verifiedevent_all_china_12years_cleaned.csv'
df = pd.read_csv(input_path)

# 确保时间列是 Datetime 对象
df['开始时间'] = pd.to_datetime(df['开始时间'])
df['结束时间'] = pd.to_datetime(df['结束时间'])

print(f"合并前事件总数: {len(df)}")

# ==========================================
# 核心合并函数
# ==========================================
def merge_city_events(group):
    # 确保组内按时间排序
    group = group.sort_values('开始时间')
    
    merged_events = []
    
    # 初始化第一个事件
    if group.empty:
        return pd.DataFrame()
        
    # 取第一行作为当前正在处理的事件
    current = group.iloc[0].to_dict()
    
    for i in range(1, len(group)):
        next_event = group.iloc[i].to_dict()
        
        # --- 判定条件 ---
        # 严格重合: next_start <= current_end
        # 允许间隔1天: next_start <= current_end + 1天
        # 这里使用严格重合或相邻（gap=0），即只要时间线连上了就算一个
        # 如果你想把中间断了一天的也连起来，把 days=0 改成 days=1
        threshold_date = current['结束时间'] + timedelta(days=2) 
        
        if next_event['开始时间'] <= threshold_date:
            # === 执行合并 ===
            # 1. 更新结束时间 (取两者较晚的)
            current['结束时间'] = max(current['结束时间'], next_event['结束时间'])
            
            # 2. 累加热度 (合并后总热度增加)
            current['总热度'] += next_event['总热度']
            
            # 3. 更新峰值热度 (取两者中更热的)
            current['峰值热度'] = max(current['峰值热度'], next_event['峰值热度'])
            
            # 4. 更新降水 (取两者中雨最大的)
            current['Max_Rainfall'] = max(current['Max_Rainfall'], next_event['Max_Rainfall'])
            
            # 5. Verified 状态 (只要有一个是True，合并后就是True)
            current['Verified'] = current['Verified'] or next_event['Verified']
            
        else:
            # === 没有重合，存档上一事件，开始新事件 ===
            merged_events.append(current)
            current = next_event
            
    # 把最后一个事件加进去
    merged_events.append(current)
    
    return pd.DataFrame(merged_events)

# ==========================================
# 执行操作
# ==========================================

# 1. 按城市分组应用合并算法
print("正在合并重复/重叠事件...")
df_merged = df.groupby('城市', group_keys=False).apply(merge_city_events)

# 2. 全局重新排序 (合并后可能会乱，再次按开始时间排)
df_merged = df_merged.sort_values(['开始时间', '城市']).reset_index(drop=True)

# 3. 重新计算“持续天数” (合并后天数可能变了)
df_merged['持续天数'] = (df_merged['结束时间'] - df_merged['开始时间']).dt.days + 1

# 4. 重新生成 Event_ID 和 Event_Code
df_merged['Event_ID'] = df_merged.index + 1
df_merged['Event_Code'] = df_merged.apply(
    lambda x: f"{x['开始时间'].strftime('%Y%m%d')}_{str(x['城市']).split(' ')[-1]}", 
    axis=1
)

# ==========================================
# 输出结果
# ==========================================
print("-" * 30)
print(f"合并后事件总数: {len(df_merged)}")
print(f"成功合并减少了: {len(df) - len(df_merged)} 条冗余记录")
print("-" * 30)

# 打印合并后的南昌市例子看看 (如果有的话)
check_city = "江西省 南昌市"
if check_city in df_merged['城市'].values:
    print(f"\n检查 {check_city} 的合并结果:")
    print(df_merged[df_merged['城市'] == check_city][['开始时间', '结束时间', '总热度']])

# 保存
output_path = r'D:\SM\zdxrun\datatest\event\all_china_12years_merge.csv'
df_merged.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"\n最终清洗版数据已保存至: {output_path}")

合并前事件总数: 7631
正在合并重复/重叠事件...


C:\Users\admin\AppData\Local\Temp\ipykernel_55336\1449354378.py:73: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_merged = df.groupby('城市', group_keys=False).apply(merge_city_events)


------------------------------
合并后事件总数: 7376
成功合并减少了: 255 条冗余记录
------------------------------

检查 江西省 南昌市 的合并结果:
           开始时间       结束时间   总热度
10   2012-05-12 2012-05-12   103
295  2012-08-22 2012-08-22   247
367  2013-05-15 2013-05-15   131
472  2013-06-28 2013-06-29   204
745  2014-05-09 2014-05-09   195
821  2014-06-19 2014-06-20   242
957  2014-07-24 2014-07-24    77
1151 2015-05-08 2015-05-08    64
1160 2015-05-14 2015-05-14    82
1280 2015-06-22 2015-06-22   223
1655 2016-06-02 2016-06-03   513
2240 2017-06-01 2017-06-01    65
2358 2017-06-23 2017-06-26   393
2397 2017-06-30 2017-07-01   129
2840 2018-03-04 2018-03-05   703
2871 2018-04-13 2018-04-13   436
3626 2019-03-21 2019-03-21   699
3926 2019-07-04 2019-07-04   370
3993 2019-07-13 2019-07-13  1163
4723 2020-07-07 2020-07-11  3938
5149 2021-03-31 2021-04-01  1283
5195 2021-05-10 2021-05-13  1101
5278 2021-05-23 2021-05-23   348
5338 2021-06-10 2021-06-10   358
6172 2022-06-19 2022-06-19   217
6266 2022-06-29 2022-06-30  

# 微博提取

In [10]:
import polars as pl
import os
import time

# ================= 配置区域 =================

# 1. 事件列表文件 (Step 6 生成的验证结果)
EVENT_FILE = r'D:\SM\zdxrun\datatest\event\all_china_12years_merge.csv'

# 2. 微博源数据文件夹 (存放 shendata20xx_geonew.csv 的目录)
SOURCE_DIR = r'D:\SM\zdxrun\datatest\city'

# 3. 输出文件 (提取出的所有原始微博)
OUTPUT_FILE = r'D:\SM\zdxrun\datatest\event_weibo_new\all_flood_weibos_12years.csv'

# 年份范围
START_YEAR = 2012
END_YEAR = 2023

# ===========================================

def normalize_loose(col_expr):
    """
    【高召回率】地点标准化辅助函数
    必须与 Step 5 的逻辑保持一致，否则匹配不上
    逻辑：保留单字地名 (如 "北京"), 多级地名取前两级
    """
    list_expr = col_expr.str.split(" ")
    return (
        pl.when(list_expr.list.len() >= 2)
        .then(list_expr.list.slice(0, 2).list.join(" "))
        .otherwise(list_expr.list.first()) # 只有一级直接保留
    )

def extract_weibos_batch():
    start_total = time.time()
    print("🚀 开始批量提取原始微博 (2012-2023)...")
    
    # -------------------------------------------------------
    # 1. 读取并预处理事件表
    # -------------------------------------------------------
    print(f"正在读取事件列表: {EVENT_FILE}")
    if not os.path.exists(EVENT_FILE):
        print("❌ 错误: 找不到事件文件")
        return

    # 读取事件表
    events_df = pl.read_csv(EVENT_FILE, infer_schema_length=0)
    
    # 数据类型转换 & 生成 Event_ID (如果表里没有的话)
    events_df = events_df.with_columns([
        pl.col("开始时间").str.to_date(strict=False),
        pl.col("结束时间").str.to_date(strict=False),
    ])

    # 如果没有 Event_ID，自动生成一个全局唯一的 ID
    if "Event_ID" not in events_df.columns:
        print("提示: 自动生成 Event_ID...")
        events_df = events_df.with_row_index("Event_ID", offset=1)

    # -------------------------------------------------------
    # 2. 逐年处理
    # -------------------------------------------------------
    all_extracted_list = []

    for year in range(START_YEAR, END_YEAR + 1):
        print(f"\n>> 正在处理 {year} 年...")
        
        # A. 筛选出当年的事件 (优化性能，只匹配当年的)
        current_year_events = events_df.filter(
            pl.col("开始时间").dt.year() == year
        )
        
        if current_year_events.is_empty():
            print(f"   [跳过] {year} 年没有记录到的事件")
            continue

        # B. 构造源文件路径
        source_file = os.path.join(SOURCE_DIR, f"shendata{year}_geonew.csv")
        if not os.path.exists(source_file):
            print(f"   [跳过] 源文件不存在: {source_file}")
            continue

        try:
            # C. 懒加载源数据
            q_source = pl.scan_csv(source_file, infer_schema_length=0, ignore_errors=True)
            
            # D. 执行匹配逻辑 (Lazy Execution)
            # 1. 转换源数据时间
            # 2. 拆分多地点
            # 3. 归一化地点
            # 4. Join 事件表
            # 5. 过滤时间范围
            
            q_result = (
                q_source
                .with_columns(
                    pl.col("发布时间").str.to_datetime(strict=False).dt.date().alias("post_date")
                )
                # 拆分 Step 4 的多地点字符串
                .with_columns(pl.col("标准化地点").str.split("; ")).explode("标准化地点")
                # 【关键】使用高召回率归一化
                .with_columns(
                    normalize_loose(pl.col("标准化地点")).alias("match_city")
                )
                .filter(pl.col("match_city").is_not_null())
                
                # Join: 将源数据与当年的事件表关联
                .join(
                    current_year_events.lazy(), 
                    left_on="match_city", 
                    right_on="城市", 
                    how="inner"
                )
                
                # Filter: 发布时间必须在事件的 [开始, 结束] 范围内
                .filter(
                    (pl.col("post_date") >= pl.col("开始时间")) & 
                    (pl.col("post_date") <= pl.col("结束时间"))
                )
                
                # Select: 选择需要的列
                .select([
                    pl.col("Event_ID"),
                    pl.col("match_city").alias("城市"),
                    pl.col("Verified").alias("气象验证结果"), # 带上验证结果
                    pl.col("Max_Rainfall").alias("最大降雨量"),
                    pl.col("开始时间"),
                    pl.col("结束时间"),
                    pl.col("发布时间"),
                    pl.col("id").alias("微博ID"),
                    pl.col("微博正文")
                ])
                
                # 去重: 防止因为一条微博有多个地点被拆分后重复提取
                .unique(subset=["微博ID", "Event_ID"])
            )
            
            # E. 收集结果
            df_year_result = q_result.collect()
            count = len(df_year_result)
            
            if count > 0:
                print(f"   [成功] 提取到 {count} 条相关微博")
                all_extracted_list.append(df_year_result)
            else:
                print(f"   [提示] 未匹配到微博内容")

        except Exception as e:
            print(f"   [错误] 处理 {year} 年时发生异常: {e}")

    # -------------------------------------------------------
    # 3. 合并与保存
    # -------------------------------------------------------
    print("\n" + "="*40)
    if all_extracted_list:
        print("正在合并所有年份数据...")
        final_df = pl.concat(all_extracted_list)
        
        # 排序: 按事件ID -> 发布时间
        final_df = final_df.sort(["Event_ID", "发布时间"])
        
        # 自动创建目录
        os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)
        
        print(f"正在保存至: {OUTPUT_FILE}")
        # 【解决Excel乱码】写入 BOM
        with open(OUTPUT_FILE, "wb") as f:
            f.write(b"\xef\xbb\xbf")
            final_df.write_csv(f)
            
        print(f"✅ 全部完成！")
        print(f"共提取原始微博: {len(final_df)} 条")
        
        # 预览
        print("\n【结果预览】")
        print(final_df.head(5))
        
    else:
        print("❌ 未提取到任何数据，请检查事件表和源文件是否匹配。")
    
    print(f"总耗时: {time.time() - start_total:.2f} 秒")

if __name__ == "__main__":
    extract_weibos_batch()

🚀 开始批量提取原始微博 (2012-2023)...
正在读取事件列表: D:\SM\zdxrun\datatest\event\all_china_12years_merge.csv

>> 正在处理 2012 年...
   [成功] 提取到 24235 条相关微博

>> 正在处理 2013 年...
   [成功] 提取到 30046 条相关微博

>> 正在处理 2014 年...
   [成功] 提取到 27723 条相关微博

>> 正在处理 2015 年...
   [成功] 提取到 35118 条相关微博

>> 正在处理 2016 年...
   [成功] 提取到 74523 条相关微博

>> 正在处理 2017 年...
   [成功] 提取到 68925 条相关微博

>> 正在处理 2018 年...
   [成功] 提取到 219866 条相关微博

>> 正在处理 2019 年...
   [成功] 提取到 262854 条相关微博

>> 正在处理 2020 年...
   [成功] 提取到 251346 条相关微博

>> 正在处理 2021 年...
   [成功] 提取到 395025 条相关微博

>> 正在处理 2022 年...
   [成功] 提取到 203702 条相关微博

>> 正在处理 2023 年...
   [成功] 提取到 310423 条相关微博

正在合并所有年份数据...
正在保存至: D:\SM\zdxrun\datatest\event_weibo_new\all_flood_weibos_12years.csv
✅ 全部完成！
共提取原始微博: 1903786 条

【结果预览】
shape: (5, 9)
┌──────────┬────────┬──────────────┬─────────────────┬───┬────────────┬────────────┬─────────────────┬─────────────────┐
│ Event_ID ┆ 城市   ┆ 气象验证结果 ┆ 最大降雨量      ┆ … ┆ 结束时间   ┆ 发布时间   ┆ 微博ID          ┆ 微博正文        │
│ ---      ┆ ---    ┆ ---      

# 文本截取

In [12]:
import polars as pl
import re
import time
import os

# ================= 配置区域 =================

# 输入 Step 7 生成的全量微博文件
INPUT_FILE = r'D:\SM\zdxrun\datatest\event_weibo_new\all_flood_weibos_12years.csv'

# 输出文件
OUTPUT_FILE = r'D:\SM\zdxrun\datatest\event_weibo_new\all_flood_weibos_12years_truncated.csv'

CONTENT_COL = '微博正文'
EVENT_ID_COL = 'Event_ID'

# 【关键修改】恢复全量关键词，与 Step 5/7 保持一致
# 否则只包含"雨"但不包含"淹"的微博会被误删
KEYWORDS = [
        '洪灾', '洪水', '大水', '洪流', '水流', '河水', '淹水', '水灾',
    '水淹', '水浸', '水涨', '趟水', '进水', '排水', '洪涝', '涝灾',
    '洪峰', '内涝', '暴雨成灾', '雨灾', '山洪爆发', '山洪',
    '水深', '齐腰', '冲毁', '冲走', '积水'
]

# 截取参数
WINDOW_SIZE = 50   # 关键词前后保留字数 (微博较短，建议由100改为50)
MAX_LEN = 500      # 最大保留长度 (超过这个长度才截取，否则不处理)

# ===========================================

# 预编译正则 (长词优先)
SORTED_KEYWORDS = sorted(set(KEYWORDS), key=len, reverse=True)
PATTERN = re.compile('|'.join(map(re.escape, SORTED_KEYWORDS)))

def smart_truncate_optimized(text):
    """
    修正后的逻辑：
    1. 【必须】关键词过滤：如果不包含任何洪涝关键词，直接扔掉。
    2. 【优化】长度判断：如果包含关键词且较短，直接返回全文。
    3. 【截取】长度判断：如果包含关键词且超长，进行截取。
    """
    if text is None:
        return None
    
    # ---------------------------------------------------------
    # 1. 第一道防线：关键词检查 (解决你的顾虑)
    # ---------------------------------------------------------
    # search 比 finditer 快，只要找到一个就行
    if not PATTERN.search(text):
        return None  # 返回 None，稍后会被 filter 删掉

    # ---------------------------------------------------------
    # 2. 第二道防线：长度优化
    # ---------------------------------------------------------
    # 代码运行到这里，说明 text 里一定包含关键词
    # 如果是短文本，直接保留全文，不用费劲去算窗口位置
    if len(text) <= MAX_LEN:
        return text

    # ---------------------------------------------------------
    # 3. 长文本截取逻辑 (只有超长文本才会走到这里)
    # ---------------------------------------------------------
    spans = []
    for match in PATTERN.finditer(text):
        start = max(0, match.start() - WINDOW_SIZE)
        end = min(len(text), match.end() + WINDOW_SIZE)
        spans.append((start, end))

    # 理论上上面 search 过了这里 spans 不会为空，但为了健壮性
    if not spans:
        return text[:MAX_LEN]

    # 合并重叠窗口
    spans.sort()
    merged = []
    if spans:
        curr_start, curr_end = spans[0]
        for next_start, next_end in spans[1:]:
            if next_start <= curr_end:
                curr_end = max(curr_end, next_end)
            else:
                merged.append((curr_start, curr_end))
                curr_start, curr_end = next_start, next_end
        merged.append((curr_start, curr_end))

    # 拼接
    result_parts = []
    total_len = 0
    for start, end in merged:
        seg_len = end - start
        if total_len + seg_len > MAX_LEN:
            remaining = MAX_LEN - total_len
            if remaining > 5:
                result_parts.append(text[start:start + remaining])
            break
        result_parts.append(text[start:end])
        total_len += seg_len

    return " ... ".join(result_parts)

def clean_and_deduplicate():
    start_time = time.time()
    print(f"🚀 启动 Step 8 (智能截取 & 去重)...")
    
    # 1. 读取数据
    if not os.path.exists(INPUT_FILE):
        print(f"❌ 文件不存在: {INPUT_FILE}")
        return

    print("正在读取数据...")
    df = pl.read_csv(INPUT_FILE, infer_schema_length=0, ignore_errors=True)
    original_count = len(df)
    print(f"原始行数: {original_count}")

    # 2. 智能截取 (使用 map_elements 应用 Python 函数)
    # 注意：Polars 处理字符串很快，但调 Python 正则较慢，这里的长度检查优化至关重要
    print("正在进行长文本智能截取...")
    
    df = df.with_columns(
        pl.col(CONTENT_COL).map_elements(smart_truncate_optimized, return_dtype=pl.Utf8).alias(CONTENT_COL)
    )

    # 3. 过滤空文本
    df = df.filter(
        pl.col(CONTENT_COL).is_not_null() & 
        (pl.col(CONTENT_COL).str.strip_chars() != "")
    )
    after_truncate_count = len(df)
    print(f"截取后保留行数: {after_truncate_count} (丢弃空行: {original_count - after_truncate_count})")

    # 4. 高级去重 (Event_ID + 文本指纹)
    print("正在生成指纹并去重...")
    
    # 生成指纹：只保留汉字和数字，去除标点符号，防止 "下雨了。" 和 "下雨了" 被算作不同
    # 正则替换：[^a-zA-Z0-9\u4e00-\u9fa5] -> ""
    df = df.with_columns(
        pl.col(CONTENT_COL)
        .str.replace_all(r"[^\w\u4e00-\u9fa5]", "") # Polars 的正则替换
        .alias("dedup_fingerprint")
    )

    # 执行去重：同一个事件下，如果指纹相同，只保留一条
    df = df.unique(subset=[EVENT_ID_COL, "dedup_fingerprint"], keep="first")
    
    # 删掉指纹列，清理内存
    df = df.drop("dedup_fingerprint")

    final_count = len(df)
    print(f"最终行数: {final_count}")
    print(f"去重删除: {after_truncate_count - final_count} 条")

    # 5. 保存
    print(f"正在保存至: {OUTPUT_FILE}")
    os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)
    
    with open(OUTPUT_FILE, "wb") as f:
        f.write(b"\xef\xbb\xbf") # BOM
        df.write_csv(f)

    print(f"✅ 处理完成！耗时: {time.time() - start_time:.2f} 秒")

if __name__ == "__main__":
    clean_and_deduplicate()

🚀 启动 Step 8 (智能截取 & 去重)...
正在读取数据...
原始行数: 1903786
正在进行长文本智能截取...
截取后保留行数: 520453 (丢弃空行: 1383333)
正在生成指纹并去重...
最终行数: 501923
去重删除: 18530 条
正在保存至: D:\SM\zdxrun\datatest\event_weibo_new\all_flood_weibos_12years_truncated.csv
✅ 处理完成！耗时: 7.19 秒


In [5]:
import pandas as pd

# 1. 文件路径（请确保 input_file 是你未被覆盖过的最原始的文件）
input_file = r'D:\SM\zdxrun\datatest\event_weibo_new\all_flood_weibos_12years_truncated.csv'
output_file = r'D:\SM\zdxrun\datatest\event_weibo_new\all_flood_weibos_12years_fixed.csv'

print("正在安全读取文件...")
# 强制按纯文本读取，保留最真实的原始面貌
df = pd.read_csv(input_file, encoding='utf-8', dtype={'发布时间': str})

# 2. 安全清洗：去掉两端空格，并用正则安全地切掉末尾的 .000 毫秒
# 使用正则表达式只替换末尾的小数点和数字，防止误伤正常数据
df['清理后时间'] = df['发布时间'].str.strip().str.replace(r'\.\d+$', '', regex=True)

print("正在进行无损时间解析...")
# 3. 尝试解析时间
parsed_dates = pd.to_datetime(df['清理后时间'], errors='coerce')

# 4. 【核心安全机制】只覆盖解析成功的，保留解析失败的原文
# 找出成功被解析为时间格式的行
success_mask = parsed_dates.notna()

# 对于成功的行，格式化为标准 YYYY-MM-DD HH:MM:SS
df.loc[success_mask, '最终发布时间'] = parsed_dates[success_mask].dt.strftime('%Y-%m-%d %H:%M:%S')

# 对于失败的行（依然无法识别的残缺数据或乱码），原样保留其清理后的文本，绝不变成空白！
df.loc[~success_mask, '最终发布时间'] = df.loc[~success_mask, '清理后时间']

# 用最终结果替换原始列，并清理临时列
df['发布时间'] = df['最终发布时间']
df = df.drop(columns=['清理后时间', '最终发布时间'])

print(f"\n--- 安全修复完成 ---")
print(f"成功标准化了 {success_mask.sum()} 条数据。")
print(f"保留了 {(~success_mask).sum()} 条原始异常数据（未被删除，原样保留以供检查）。")

print("\n正在保存文件...")
# 保存文件
df.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"处理完成！安全文件已保存至：\n{output_file}")

正在安全读取文件...
正在进行无损时间解析...

--- 安全修复完成 ---
成功标准化了 456265 条数据。
保留了 45658 条原始异常数据（未被删除，原样保留以供检查）。

正在保存文件...
处理完成！安全文件已保存至：
D:\SM\zdxrun\datatest\event_weibo_new\all_flood_weibos_12years_fixed.csv


In [6]:
import pandas as pd

# 读取你刚刚修复好的那个安全文件
safe_file = r'D:\SM\zdxrun\datatest\event_weibo_new\all_flood_weibos_12years_fixed.csv'
output_bad_file = r'D:\SM\zdxrun\datatest\event_weibo_new\only_45658_bad_data.csv'

print("正在扫描文件...")
df = pd.read_csv(safe_file, encoding='utf-8', dtype=str)

# 找出依然无法被识别为时间的行
parsed = pd.to_datetime(df['发布时间'], errors='coerce')
bad_mask = parsed.isna()

# 提取这些坏数据
df_bad = df[bad_mask]

print(f"抓到了 {len(df_bad)} 条真正异常的数据！")

if len(df_bad) > 0:
    print("\n下面是前 10 条真正的异常内容：")
    for idx, val in df_bad['发布时间'].head(10).items():
        print(f"原始行号大致在 {idx+2}: 【{val}】")
        
    df_bad.to_csv(output_bad_file, index=False, encoding='utf-8-sig')
    print(f"\n我已经把这 {len(df_bad)} 条数据单独打包存到了：\n{output_bad_file}")
    print("👉 请直接打开这个文件，看一眼你就全明白了！")

正在扫描文件...
抓到了 0 条真正异常的数据！


# llm提取

In [ ]:
import os
import json
import re
import ast  # <--- 新增
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
from openai import OpenAI

# ==============================================================================
# 1. 配置区域
# ==============================================================================
SERVER_IP = "192.168.3.10" 
SERVER_PORT = "8000"

BASE_URL = f"http://{SERVER_IP}:{SERVER_PORT}/v1"
API_KEY = "EMPTY"

VLLM_MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4"

# 文件路径
INPUT_FILE = r"D:\SM\zdxrun\datatest\event_weibo_new\all_flood_weibos_12years_truncated.csv"

OUTPUT_FILE = r"D:\SM\zdxrun\datatest\llm结果\all city\extract0212.csv" 
CONTENT_COL = '微博正文'

# 性能参数
MAX_WORKERS = 60      
CHUNK_SIZE = 2000     

# ==============================================================================
# 2. Prompt 设计
# ==============================================================================
SYSTEM_PROMPT = """
你是一个灾害信息提取专家。请分析微博文本，判断是否发生了“内涝/积水”灾害，并提取关键信息。

【提取规则 (严格遵守)】

1. 事件判断 (is_w)
   目标：识别所有与“暴雨灾害”及“内涝积水”相关的实际发生事件。
   
   【判定为 True（是）的标准】：
   只要满足以下任一条件，即判定为 True：
   A. 积水与内涝：
      - 任何场所（道路、隧道、高架、地铁站、地下室、小区、商场等）出现积水或被淹。
      - 描述水位：如“水深及腰”、“没过轮胎”、“淹到一楼”、“积水严重”等。
   B. 交通与设施受阻：
      - 因雨水导致的交通中断、车辆熄火、车辆漂浮、无法通行、断路。
      - 公共设施受灾：如“地铁倒灌”、“公交停运（因积水）”、“桥梁断裂”、“道路塌方”。
   C. 人员受困、求助、失联等：
      - 明确提及“被困”、“冲走”、“失联”、“救援”，或暗示周围已被水包围。
   D. 隐喻与视觉描述：
      - 使用“看海”、“开船”、“游泳”、“瀑布（形容楼梯流水）”、“变河了”等夸张描述。
   
   【判定为 False（否）的标准】：
   - 仅当文本是纯粹的“天气预报”、“预警通知”或“祈福加油”，且完全没有提及任何具体受灾或救援现场时，才为 False。

2. 提取规则
若 is_w 为 True，请提取以下信息：
   -时间 (tm)：提取文本中描述的内涝事件发生时间。如果文本中未出现任何与事件发生相关的时间信息，则 tm 置为空字符串 ""。
   -城市识别 (city)：提取内涝发生的地级市名称（如“北京市”“武汉市”）。无法确定城市时，city 置为空字符串。
   -事件(evs)：
        -完整地址 (loc)：直接摘录原文中出现的最完整地点描述，优先包含道路名称、小区名、地标建筑等信息。
        若有多个地名写在一起但用“、”或“，”隔开时，需拆分成独立的地址。如果是**交叉口**（如“A路与B路交叉口”），视为**一个地点**，不可拆分。
        -受灾描述 (stat)：必须从原文中直接摘录对积水/受灾状况的描述和影响，包括数值或形容性表达，如“50厘米”“没过膝盖”“整条路看海了”等，以及“交通瘫痪”“人被冲走”等。
        提炼关键词，不要整句摘录，禁止自行总结、推断或改写原意。
        - 拆分逻辑：若文中出现顿号“、”或逗号“，”分隔的**多个独立地点**（如“A路、B路、C路”），**必须生成多个 evs 对象**，不可合并为一个。
3. 一一对应
   若文本中描述了多处不同地点的积水情况，需生成多个事件对象，并保证每个 city、loc、stat、tm之间一一对应，不许偷懒。

4. 输出约束
   - 输出必须是标准的 JSON 格式。
   - 若“is_w”为False，则其他信息（city、tm、loc、stat）全为空值。
   
【范例1】
{
    "is_w": true,
    "tm": "下午3点",
    "evs": [
        {
            "city": "郑州市",
            "loc": "陇海路",
            "stat": "车辆漂浮"
        }
    ]
}

【范例2：列表拆分（顿号处理）】


输入：7月20日，受暴雨影响，西安昨晚迎短时大雨，由于管网排水不及，市政管辖范围内油库街、文景路下穿、红旗东路积水严重。
输出：
{
    "is_w": true,
    "tm": "7月20日",
    "evs": [
        {"city": "西安市", "loc": "油库街", "stat": "积水严重"},
        {"city": "西安市", "loc": "文景路下穿", "stat": "积水严重"},
        {"city": "西安市", "loc": "红旗东路", "stat": "积水严重"}
    ]
}

若判定为 False：
{
    "is_w": false,
    "tm": "",
    "evs": []
}
"""

# ==============================================================================
# 3. 核心提取逻辑
# ==============================================================================
client = OpenAI(base_url=BASE_URL, api_key=API_KEY)


def extract_json_from_text(text):
    """
    终极 JSON 提取器 v2.0：
    1. 处理 Markdown 包裹
    2. 修复 broken string (例如: "A" 和 "B" -> "A, B") 【新增】
    3. 修复 keys 缺失引号 (例如: city: "v" -> "city": "v")
    4. 自动截断尾部废话
    """
    text = text.strip()
    
    # === 1. 提取核心文本 ===
    match = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
    if match:
        text = match.group(1)
    else:
        start = text.find('{')
        if start != -1:
            text = text[start:]
        else:
            return None 

    # === 2. 针对性修复逻辑 (按顺序执行) ===
    
    # 【新增修复】: 缝合被 "和"、"、" 切断的字符串
    # 匹配模式: (引号)(空格)(和/、/以及)(空格)(引号) -> 替换为 ", "
    # 例子: "洪水" 和 "积水"  -->  "洪水, 积水"
    text = re.sub(r'"\s*(?:和|、|以及)\s*"', ', ', text)

    # 【原有修复】: 给丢失引号的 Key 加上引号
    # 例子: city: "北京" --> "city": "北京"
    text = re.sub(r'([,{]\s*)(\w+)(\s*:)', r'\1"\2"\3', text)

    # === 3. 尝试解析 ===
    try:
        # 3.1 优先尝试标准解析
        obj, _ = json.JSONDecoder().raw_decode(text)
        return obj
    except:
        pass

    try:
        # 3.2 尝试 Python eval (处理单引号)
        # 尽量找到最后一个 } 来截断
        end = text.rfind('}')
        if end != -1:
            potential_dict = text[:end+1]
            return ast.literal_eval(potential_dict)
    except:
        pass

    return None

def extract_info_row(row_data):
    text = str(row_data.get(CONTENT_COL, ""))
    error_res = row_data.copy()
    
    if len(text) < 5 or text.lower() == 'nan':
        error_res.update({"llm_processed": False, "error": "Empty text"})
        return [error_res]

    try:
        response = client.chat.completions.create(
            model=VLLM_MODEL_NAME,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"文本：\n{text}"}
            ],
            temperature=0,
            max_tokens=4096, 
        )
        
        raw_output = response.choices[0].message.content.strip()
        
        # === 【修改点】使用新的解析函数 ===
        data = extract_json_from_text(raw_output)

        if not data:
            # 如果还是失败，记录 raw_output 以便排查
            error_res.update({"is_w": False, "error": "JSON Decode Fail", "raw_output": raw_output})
            return [error_res]

        # === 结果构建 (保持不变) ===
        result_rows = []
        common_time = data.get("tm", "")
        is_water = data.get("is_w", False)
        
        # 兼容 True/False 字符串或布尔值
        is_w_bool = False
        if isinstance(is_water, bool):
            is_w_bool = is_water
        elif str(is_water).lower() in ['true', '1', 'yes']:
            is_w_bool = True

        if not is_w_bool:
            res = row_data.copy()
            res.update({
                "llm_extracted_time": common_time,
                "is_waterlogging": False,
                "llm_city": "", "llm_location": "", "llm_water_status": "",
                "error": None
            })
            result_rows.append(res)
        else:
            events = data.get("evs", [])
            if not events:
                res = row_data.copy()
                res.update({
                    "llm_extracted_time": common_time,
                    "is_waterlogging": True,
                    "llm_city": "", "llm_location": "原文未提及具体位置", "llm_water_status": "",
                    "error": None
                })
                result_rows.append(res)
            else:
                for event in events:
                    res = row_data.copy()
                    res.update({
                        "llm_extracted_time": common_time,
                        "is_waterlogging": True,
                        "llm_city": event.get("city", ""),
                        "llm_location": event.get("loc", ""),   
                        "llm_water_status": event.get("stat", ""),
                        "error": None
                    })
                    result_rows.append(res)

        return result_rows

    except Exception as e:
        error_res.update({"error": str(e)[:100], "raw_output": ""})
        return [error_res]

# ==============================================================================
# 4. 主流程
# ==============================================================================
# ==============================================================================
# 4. 主流程 (已修复编码问题)
# ==============================================================================
def main():
    if not os.path.exists(INPUT_FILE):
        print(f"错误：找不到输入文件 {INPUT_FILE}")
        return

    # --- 修改点1：自动检测或指定正确的中文编码 ---
    # 优先尝试 utf-8-sig，失败则尝试 gb18030 (兼容 gbk)
    detect_encodings = ['utf-8-sig', 'gb18030', 'gbk', 'big5']
    final_encoding = 'utf-8-sig' # 默认
    
    print("正在检测文件编码...")
    for enc in detect_encodings:
        try:
            # 尝试读取前几行
            pd.read_csv(INPUT_FILE, nrows=5, encoding=enc)
            final_encoding = enc
            print(f">> 成功检测到文件编码: {enc}")
            break
        except UnicodeDecodeError:
            continue
        except Exception as e:
            # 如果是其他错误（如文件格式错），可能不是编码问题，但也跳过
            print(f"尝试编码 {enc} 失败: {e}")
            continue
            
    # ------------------------------------------------

    if os.path.exists(OUTPUT_FILE):
        print(f"警告：输出文件 {OUTPUT_FILE} 已存在。")
        print(">> 正在删除旧文件，确保数据格式纯净...")
        try:
            os.remove(OUTPUT_FILE)
        except OSError as e:
            print(f"删除失败: {e}，请手动删除后重试。")
            return
    
    print("正在初始化表头结构...")
    try:
        # --- 修改点2：使用检测到的编码 ---
        sample_df = pd.read_csv(INPUT_FILE, nrows=0, low_memory=False, encoding=final_encoding)
        original_cols = sample_df.columns.tolist()
    except Exception as e:
        print(f"读取输入文件失败: {e}")
        return

    new_cols = [
        "is_waterlogging", 
        "llm_extracted_time",
        "llm_city",
        "llm_location",      
        "llm_water_status", 
        "error",
        "raw_output",
        "llm_processed"
    ]
    
    FINAL_COLUMNS = original_cols + new_cols
    
    print(f"=== 开始处理 ===")
    print(f"输入: {INPUT_FILE}")
    print(f"输出: {OUTPUT_FILE}")
    print(f"线程: {MAX_WORKERS} | 编码: {final_encoding} | 增强解析: ON")

    total_processed = 0

    try:
        chunk_iterator = pd.read_csv(
            INPUT_FILE, 
            chunksize=CHUNK_SIZE, 
            low_memory=False,
            encoding=final_encoding,  # --- 修改点3：使用检测到的编码 ---
            on_bad_lines='skip' 
        )

        for i, df_chunk in enumerate(chunk_iterator):
            
            if CONTENT_COL not in df_chunk.columns:
                print(f"错误：当前块缺少 '{CONTENT_COL}' 列，跳过。")
                continue

            rows = df_chunk.to_dict(orient='records')
            chunk_results = []
            
            print(f"\n>> Chunk {i+1} ({len(rows)} 行)...")

            with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
                future_to_row = {executor.submit(extract_info_row, row): row for row in rows}
                
                for future in tqdm(as_completed(future_to_row), total=len(rows), desc="提取进度", leave=False):
                    res = future.result()
                    if res:
                        chunk_results.extend(res)

            if chunk_results:
                result_df = pd.DataFrame(chunk_results)
                
                # 核心：强制重排对齐
                result_df = result_df.reindex(columns=FINAL_COLUMNS)
                
                write_mode = 'w' if i == 0 else 'a'
                write_header = (i == 0)
                
                result_df.to_csv(
                    OUTPUT_FILE,
                    mode=write_mode,
                    index=False,
                    header=write_header,
                    encoding="utf-8-sig" # 输出文件通常保持 utf-8-sig 以便 Excel 打开
                )

            total_processed += len(rows)
            print(f"   [成功] 累计处理: {total_processed}")

    except KeyboardInterrupt:
        print("\n用户停止。")
    except Exception as e:
        print(f"\n未知错误: {e}")
    finally:
        print("任务结束。")

if __name__ == "__main__":
    main()

# llm结果处理

In [19]:
import pandas as pd

# 1. 读取 CSV (建议保持 dtype=str 以防止 pandas 自动转换类型导致混乱)
input_file = r'D:\SM\zdxrun\datatest\llm结果\all city\extract0208.csv'
df = pd.read_csv(input_file, dtype=str, encoding='utf-8-sig')

# 2. 【核心修改】标准化的筛选逻辑
# 将 is_waterlogging 列统一转为字符串 -> 去除空格 -> 转为小写 -> 检查是否包含 'true' 或 '1'
df_filtered = df[
    df["is_waterlogging"].astype(str).str.strip().str.lower().isin(["true", "1", "yes"])
]

# 3. 删除不需要的列 (errors='ignore' 很好，保持不动)
df_filtered = df_filtered.drop(columns=["original_content", "error", "raw_output"], errors="ignore")

# 4. 打印一下看看现在提取了多少行 (调试用)
print(f"原始行数: {len(df)}")
print(f"提取后行数: {len(df_filtered)}")

# 5. 输出
output_file = r"D:\SM\zdxrun\datatest\llm结果\all city\extract0208_T.csv"
df_filtered.to_csv(output_file, index=False, encoding="utf-8-sig")

# 6. 预览结果
print(df_filtered.head())

原始行数: 734979
提取后行数: 488308
   Event_ID       城市 气象验证结果               最大降雨量        开始时间        结束时间  \
38     3994  江西省 南昌市   True  143.22891843786746  2019-07-13  2019-07-13   
39     5551  河南省 郑州市   True   446.7344237713874  2021-07-20  2021-07-23   
40      150      北京市   True  168.72454248519682  2012-07-22  2012-07-25   
41     6466  河南省 郑州市   True    89.2001310765404  2022-07-20  2022-07-22   
42     3529  广东省 汕头市   True   255.2813728187629  2018-08-31  2018-09-03   

                       发布时间              微博ID  \
38         2019-07-13 10:17  4393536693031203   
39         2021-07-23 21:43  4662238797369830   
40         2012-07-23 09:27  3470884730012937   
41  2022-07-22 19:13:00.000  4794110160406015   
42         2018-09-01 16:54  4279484566419664   

                                                 微博正文 is_waterlogging  \
38                            #南昌暴雨积水#🙏🙏🙏翻滚吧L陈奕天的微博视频            True   
39   这么严重的洪灾，不到2天，有的地方的水还没有排掉，郑州供电竟然恢复八成，供水恢复九成，太厉害了！            True   
40      

# 城市匹配

In [20]:
import pandas as pd
import os

# ================= 配置区域 =================
INPUT_FILE = r'D:\SM\zdxrun\datatest\llm结果\all city\extract0208_T.csv'
DIFF_FILE = r'D:\SM\zdxrun\datatest\llm结果\all city\diff_check.csv'
CLEAN_FILE = r'D:\SM\zdxrun\datatest\llm结果\all city\clean_data.csv'

# ===========================================

def normalize_city(text):
    """简单的城市名清洗：去除省、市、区、县后缀，便于比较"""
    if not isinstance(text, str):
        return ""
    # 去除常见行政后缀，保留核心地名
    # 注意：顺序很重要，先长后短
    for suffix in ['特别行政区', '自治州', '地区', '盟', '省', '市', '区', '县']:
        if text.endswith(suffix) and len(text) > len(suffix): # 防止把"县"字本身删没了
            return text[:-len(suffix)]
    return text

def is_city_match(row):
    """
    判断逻辑：
    1. 如果 LLM 没提取出城市 -> 视为不匹配 (根据你的需求保留 False)
    2. 如果核心地名一致 -> True
    3. 如果是双向包含 -> True
    """
    # 1. 获取原始值
    raw_city = str(row.get('城市', '')).strip()
    llm_city = str(row.get('llm_city', '')).strip()
    
    # 2. 如果 LLM 为空，直接返回 False (视为提取失败/不匹配)
    if not llm_city or llm_city.lower() == 'nan':
        return False
        
    # 3. 标准化 (去除 "市", "省" 等后缀)
    # 比如 "北京市" -> "北京", "北京" -> "北京"
    core_raw = normalize_city(raw_city)
    core_llm = normalize_city(llm_city)
    
    # 4. 核心匹配逻辑 (双向包含)
    # 情况A: 完全相等 ("北京" == "北京")
    if core_raw == core_llm:
        return True
        
    # 情况B: 包含关系
    # raw="湖北省武汉市", llm="武汉" -> True
    # raw="武汉", llm="武汉市" -> True
    if core_llm in raw_city or core_raw in llm_city:
        return True
        
    return False

def main():
    print("🚀 开始筛选城市匹配数据...")
    
    # 1. 读取
    try:
        df = pd.read_csv(INPUT_FILE, encoding='utf-8-sig', dtype=str)
    except:
        df = pd.read_csv(INPUT_FILE, encoding='gb18030', dtype=str)
        
    print(f"原始数据量: {len(df)}")
    
    # 2. 应用筛选
    # 创建一个布尔 Series
    mask = df.apply(is_city_match, axis=1)
    
    # 3. 分割数据
    clean_df = df[mask].copy()
    diff_df = df[~mask].copy()
    
    # 4. 保存
    os.makedirs(os.path.dirname(CLEAN_FILE), exist_ok=True)
    
    clean_df.to_csv(CLEAN_FILE, index=False, encoding='utf-8-sig')
    diff_df.to_csv(DIFF_FILE, index=False, encoding='utf-8-sig')
    
    print("-" * 30)
    print(f"✅ 筛选完成")
    print(f"保留行数 (Clean): {len(clean_df)}  | 占比: {len(clean_df)/len(df):.1%}")
    print(f"剔除行数 (Diff) : {len(diff_df)}")
    print(f"文件已保存至: {CLEAN_FILE}")

if __name__ == "__main__":
    main()

🚀 开始筛选城市匹配数据...
原始数据量: 488308
------------------------------
✅ 筛选完成
保留行数 (Clean): 332687  | 占比: 68.1%
剔除行数 (Diff) : 155621
文件已保存至: D:\SM\zdxrun\datatest\llm结果\all city\clean_data.csv


# 去重

In [22]:
import pandas as pd

df = pd.read_csv(
    r'D:\SM\zdxrun\datatest\llm结果\all city\clean_data.csv',
    dtype=str
)

# 指定用于判重的列
dedup_cols = [
    "Event_ID", "发布时间","llm_extracted_time",
    "llm_location", "llm_water_status"
]

# 去重（保留第一次出现的）
df_dedup = df.drop_duplicates(
    subset=dedup_cols,
    keep="first"
)

# 【新增】删除 llm_location 为空的行
df_dedup = df_dedup[df_dedup['llm_location'].notna() & (df_dedup['llm_location'].str.strip() != '')]

# 重置索引
df_dedup = df_dedup.reset_index(drop=True)

# 保存
df_dedup.to_csv(
    r"D:\SM\zdxrun\datatest\llm结果\all city\clean_data_dedup.csv",
    index=False,
    encoding="utf-8-sig"
)

print("原始行数：", len(df))
print("去重后行数：", len(df_dedup))
print("删除空 llm_location 后行数：", len(df_dedup))
print("总共删除的行数：", len(df) - len(df_dedup))

原始行数： 332687
去重后行数： 291988
删除空 llm_location 后行数： 291988
总共删除的行数： 40699


# 地址清洗

In [27]:
import pandas as pd
import re

# ================= 配置区域 =================
INPUT_FILE = r"D:\SM\zdxrun\datatest\llm结果\all city\clean_data_dedup.csv"
OUTPUT_FILE = r"D:\SM\zdxrun\datatest\llm结果\all city\ready_for_geocoding_v2.csv"

# 1. 精确黑名单 (如果 llm_location 等于这些词，直接删)
STRICT_BLACKLIST = {
    "家", "家中", "家里", "老家", "楼下", "门口", "自家",
    "路上", "路面", "道路", "马路", "街道", "街上", "路段", "公路上",
    "小区", "社区", "院子", "车库", "地下室", "停车场",
    "市区", "城市", "城里", "市内", "城区", "郊区", "乡下",
    "多地", "多处", "到处", "部分地区", "局部", "各地", "多条道路",
    "现场", "目击者", "网友", "朋友圈", "视频", "直播", "网上",
    "积水", "内涝", "水浸", "被淹", "严重积水", "洪水",
    "这些路段", "相关路段", "上述路段", "该路段", "某些路段", "部分路段",
    "公交路线", "地铁站", "火车站", "汽车站",
    "全城", "全市", "全区", "全县", "全省", "境内", "辖区", "当地"
}

# 2. 模糊黑名单 (如果 llm_location 包含这些词，直接删)
FUZZY_BLACKLIST = [
    "未明确", "未提及", "不详", "未知", "无具体", "具体地点不详", 
    "无法确定", "没说", "不清楚", "哪里", "某地", "什么地方"
]

# ===========================================

def normalize_city_name(name):
    """把 '武汉市' -> '武汉', '湖北省武汉市' -> '武汉'"""
    if pd.isna(name): return ""
    # 简单处理：取最后两个字作为核心词 (如 '武汉')
    # 或者用 split 取最后一段
    parts = str(name).split(" ") # 你的数据是 "湖北省 武汉市"
    core = parts[-1]
    # 去除后缀
    for suffix in ['市', '区', '县', '盟', '州']:
        if core.endswith(suffix) and len(core) > len(suffix):
            return core[:-len(suffix)]
    return core

def is_valid_location(row):
    """
    判断地址是否有效 (结合城市名进行判断)
    """
    loc = str(row['llm_location']).strip()
    city = str(row['城市']).strip()
    
    # 1. 基础非空检查
    if not loc or loc.lower() == 'nan':
        return False
        
    # 2. 长度过滤 (单字地名很难定位准确)
    if len(loc) < 2:
        return False
        
    # 3. 精确黑名单过滤
    if loc in STRICT_BLACKLIST:
        return False
        
    # 4. 模糊黑名单过滤
    for bad_word in FUZZY_BLACKLIST:
        if bad_word in loc:
            return False
            
    # 5. 【核心优化】同名检测 / 包含检测
    # 如果 location 就是城市名本身 (如 城市="武汉市", loc="武汉") -> 无效
    core_city = normalize_city_name(city)
    core_loc = normalize_city_name(loc)
    
    # 情况A: loc 等于 city 核心词 (武汉 vs 武汉)
    if core_loc == core_city:
        return False
        
    # 情况B: loc 包含在 city 中 (如 "武汉市" vs "武汉")
    # 注意：防止误删 "武汉大学" (包含武汉)，所以要反过来判断
    # 逻辑：如果 city 包含了 loc 的全部内容，说明 loc 只是 city 的一部分或简称
    # 例如：city="湖北省武汉市", loc="武汉" -> True (删除)
    # 例如：city="武汉市", loc="江汉区" -> False (保留)
    if loc in city:
        # 特殊情况：如果 loc 是区县名，保留
        # 简单判断：如果 city 结尾是 loc，可能是区县 (如 "武汉市江汉区", loc="江汉区")
        if city.endswith(loc):
            return True
        return False

    return True

def make_full_address(row):
    city = str(row['城市']).strip()
    loc = str(row['llm_location']).strip()
    
    # 如果 loc 已经包含了 city (如 "武汉市江汉路")，直接返回 loc
    # 注意处理 "湖北省 武汉市" 这种中间有空格的情况
    clean_city = city.replace(" ", "")
    if clean_city in loc:
        return loc
        
    # 否则拼接 (优先使用带省份的 city)
    return f"{city}{loc}"

def clean_and_optimize():
    print("🚀 开始第二轮清洗 (去除泛指地点)...")
    
    df = pd.read_csv(INPUT_FILE, dtype=str)
    original_count = len(df)
    
    # 1. 应用筛选逻辑
    mask = df.apply(is_valid_location, axis=1)
    
    valid_df = df[mask].copy()
    invalid_df = df[~mask].copy()
    
    # 2. 生成完整地址
    if len(valid_df) > 0:
        valid_df['full_address'] = valid_df.apply(make_full_address, axis=1)
    
    # 3. 保存
    valid_df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
    
    dropped_path = OUTPUT_FILE.replace("ready_for_geocoding_v2.csv", "dropped_v2.csv")
    invalid_df.to_csv(dropped_path, index=False, encoding='utf-8-sig')

    print("-" * 30)
    print(f"原始行数: {original_count}")
    print(f"有效地址: {len(valid_df)} (已保存至 ready_for_geocoding_v2.csv)")
    print(f"清洗剔除: {len(invalid_df)} (已保存至 dropped_v2.csv)")
    print("-" * 30)
    
    if not valid_df.empty:
        print("【有效数据预览 (Top 5)】")
        # 随机抽样查看，避免只看头部
        print(valid_df[['城市', 'llm_location', 'full_address']].sample(5))

if __name__ == "__main__":
    clean_and_optimize()

🚀 开始第二轮清洗 (去除泛指地点)...
------------------------------
原始行数: 291988
有效地址: 271179 (已保存至 ready_for_geocoding_v2.csv)
清洗剔除: 20809 (已保存至 dropped_v2.csv)
------------------------------
【有效数据预览 (Top 5)】
               城市 llm_location        full_address
267663   黑龙江省 大庆市    龙凤外环路部分路段   黑龙江省 大庆市龙凤外环路部分路段
257406    广东省 汕头市      潮阳区棉北街道      广东省 汕头市潮阳区棉北街道
226439    江西省 南昌市  北京东路青山湖大道路口  江西省 南昌市北京东路青山湖大道路口
283822    湖南省 长沙市       地铁橘子洲站       湖南省 长沙市地铁橘子洲站
103135  黑龙江省 哈尔滨市         锦园别墅       黑龙江省 哈尔滨市锦园别墅


# 智能地理编码

In [11]:
import pandas as pd
import requests
import time
import os
import re
import difflib

# ================= 配置区域 =================
# 请替换为你的高德 Key

AMAP_KEY = ""

INPUT_FILE = r'D:\SM\zdxrun\datatest\llm结果\48shi\extract0119\coords\rejected_locations.csv'
OUTPUT_FILE = r'D:\SM\zdxrun\datatest\llm结果\48shi\extract0119\coords\smart_coords3_rejected.csv'

DAILY_LIMIT = 150000 
BATCH_SIZE = 10      

# ================= 核心工具函数 =================

def clean_address_text(text):
    if not text or str(text).lower() in ['nan', 'none', '']:
        return ''
    text = str(text)
    # 1. 去除括号
    text = re.sub(r'\(.*?\)|（.*?）|\[.*?\]', '', text)
    # 2. 去除楼层房号
    text = re.sub(r'\d+(?:层|楼|室|号房|F|B)', '', text)
    
    # === 3. 关键清洗：去除方位和距离描述 ===
    text = re.sub(r'往.*?方向', '', text) 
    text = re.sub(r'向[东南西北].*?米', '', text)
    text = re.sub(r'距.*?\d+米', '', text)
    # ==================================

    # 4. 去除泛词
    text = re.sub(r'(?:位于|坐落于|地处|大概|大约|附近|对面|旁|侧)', '', text)
    text = re.sub(r'[ \t\r\n,，。、]', '', text)
    return text.strip()

def calc_text_similarity(text_a, text_b):
    if not text_a or not text_b: return 0.0
    str1 = str(text_a).lower()
    str2 = str(text_b).lower()
    if len(str2) > len(str1):
        str2_core = re.sub(r'^.*?(?:省|市|区|县)', '', str2)
        if str2_core: str2 = str2_core
    return difflib.SequenceMatcher(None, str1, str2).ratio()


def api_get(url, params, max_retries=5):
    """
    改进版：增加指数退避策略
    """
    for attempt in range(max_retries):
        try:
            # 这里的超时设大一点，防止网络波动
            res = requests.get(url, params=params, timeout=8) 
            
            if res.status_code == 200:
                data = res.json()
                
                if 'status' in data and data['status'] == '0':
                    info = str(data.get('info', ''))
                    
                    # 遇到 QPS 限制
                    if 'QPS' in info or 'Too Fast' in info:
                        # 动态等待时间：第一次2秒，第二次4秒，第三次6秒...
                        wait_time = (attempt + 1) * 2
                        #print(f" [API拥堵-等待{wait_time}秒]", end="")
                        time.sleep(wait_time)
                        continue # 发起下一次重试
                    
                    elif 'DAILY_LIMIT' in info:
                        print("\n[严重] 今日额度已用完！")
                        return "LIMIT_REACHED"
                    else:
                        return None
                
                return data
            
            # 如果是 HTTP 错误（如 502/404），稍等一下再试
            time.sleep(1)
            
        except Exception:
            # 网络报错（断网等）
            time.sleep(2)
            
    return None

def get_geocode_base(address, city):
    """策略A: 地理编码"""
    search_term = address 
    url = 'https://restapi.amap.com/v3/geocode/geo'
    params = {'key': AMAP_KEY, 'address': search_term, 'city': city, 'output': 'JSON'}
    
    data = api_get(url, params)
    
    if data == "LIMIT_REACHED": return "LIMIT_REACHED", None, None, None, None, 0
    
    if data and data['status'] == '1' and int(data['count']) > 0:
        geo = data['geocodes'][0]
        count = int(data['count'])
        return geo['location'], geo['level'], geo['formatted_address'], geo['province'], geo['city'], count
    return None, None, None, None, None, 0

def get_poi_search(keyword, city):
    """策略B: 关键字搜索"""
    search_term = keyword
    if city and city not in search_term:
        search_term = f"{city}{search_term}"

    url = 'https://restapi.amap.com/v3/place/text'
    params = {'key': AMAP_KEY, 'keywords': search_term, 'city': city, 'citylimit': 'true', 'offset': 1, 'page': 1, 'output': 'JSON'}
    
    data = api_get(url, params)
    
    if data == "LIMIT_REACHED": return "LIMIT_REACHED", None, None, None, None, 0
    
    if data and data['status'] == '1' and int(data['count']) > 0:
        poi = data['pois'][0]
        count = int(data['count'])
        pname = poi.get('pname', '')
        cname = poi.get('cityname', '')
        adname = poi.get('adname', '')
        addr = poi.get('address', '')
        name = poi.get('name', '')
        if isinstance(addr, list) or not addr: addr = ''
        full_prefix = pname
        if cname and cname != pname: full_prefix += cname
        full_address = f"{full_prefix}{adname}{addr}{name}"
        return poi['location'], 'POI', full_address, pname, cname, count
        
    return None, None, None, None, None, 0

# ... (check_city_match 函数保持不变) ...

def smart_locate(row):
    # ... (前面的代码保持不变) ...
    city = str(row.get('llm_city', row.get('城市', ''))).strip()
    if city.lower() in ['nan', 'none']: city = ''
    raw_addr = str(row.get('llm_location', '')).strip()
    clean_addr = clean_address_text(raw_addr)
    
    if not clean_addr:
        return None, None, "Empty_Input", "Empty", "", "None", 0, 0.0

    low_quality_levels = ['city', 'district', 'province', '']
    
    # 阶段 1
    loc, level, fmt_addr, res_prov, res_city, count = get_geocode_base(clean_addr, city)
    if loc == "LIMIT_REACHED": return "LIMIT_REACHED", None, None, None, None, None, 0, 0
    
    # ... (中间的逻辑判断保持不变) ...
    # 为了节省篇幅，这里假设你保留了原来的逻辑结构
    # ...
    
    final_loc = None
    strategy = "Not_Found"
    final_match = ""
    final_level = "None"
    final_count = 0
    
    is_city_correct = check_city_match(res_city, res_prov, city, fmt_addr)
    
    if loc and is_city_correct and (level not in low_quality_levels):
        final_loc = loc; strategy = f"GeoCode_HighQuality"; final_match = fmt_addr; final_level = level; final_count = count
    else:
        # 阶段 2
        loc_poi, poi_level, full_poi_addr, poi_prov, poi_city, poi_count = get_poi_search(clean_addr, city)
        if loc_poi == "LIMIT_REACHED": return "LIMIT_REACHED", None, None, None, None, None, 0, 0
        
        is_poi_city_correct = check_city_match(poi_city, poi_prov, city, full_poi_addr)
        
        if loc_poi and is_poi_city_correct:
            final_loc = loc_poi; strategy = "POI_Search"; final_match = full_poi_addr; final_level = poi_level; final_count = poi_count
            if not is_city_correct and loc: strategy = "POI_Corrected"
        elif loc and is_city_correct:
            final_loc = loc; strategy = "GeoCode_LowQuality"; final_match = fmt_addr; final_level = level; final_count = count
        elif loc:
            final_loc = loc; strategy = "WRONG_CITY_DRIFT"; final_match = fmt_addr; final_level = level; final_count = count

    # ... (相似度计算保持不变) ...
    sim_score = 0.0
    if final_loc and final_match:
        sim_score = calc_text_similarity(clean_addr, final_match)
        if sim_score < 0.2 and final_level != 'No.': strategy += "_LowSim"

    if final_loc:
        lng, lat = final_loc.split(',')
        return float(lng), float(lat), strategy, final_match, clean_addr, final_level, final_count, sim_score
    else:
        return None, None, "Not_Found", "", clean_addr, "None", 0, 0.0

# ================= 主流程 (加入了缓存机制) =================

def main():
    print(f"正在读取输入文件: {INPUT_FILE} ...")
    if not os.path.exists(INPUT_FILE): return

    try:
        df_input = pd.read_csv(INPUT_FILE, encoding='utf-8-sig')
    except UnicodeDecodeError:
        df_input = pd.read_csv(INPUT_FILE, encoding='gb18030')
    
    # === 新增：定义缓存字典 ===
    # 格式: { (city, address): (result_tuple) }
    addr_cache = {}
    
    start_index = 0
    if os.path.exists(OUTPUT_FILE):
        try:
            df_done = pd.read_csv(OUTPUT_FILE)
            start_index = len(df_done)
            print(f"检测到断点，从第 {start_index} 行继续...")
            
            # (可选) 如果你想更完美，可以把已处理的结果也读入缓存，
            # 这样如果后面还有重复的，就能直接用。但为了启动速度，这里暂时只在运行时缓存。
        except: pass
            
    rows_to_process = df_input.iloc[start_index:]
    total_rows = len(rows_to_process)
    print(f"剩余任务: {total_rows} 条")
    
    batch_buffer = []
    count = 0
    cache_hits = 0 # 统计缓存命中次数
    
    for index, row in rows_to_process.iterrows():
        if count >= DAILY_LIMIT: break
        
        # 1. 准备缓存 Key
        city_key = str(row.get('llm_city', row.get('城市', ''))).strip()
        addr_key = str(row.get('llm_location', '')).strip()
        # 用清洗后的地址做 Key 会更准，但在 smart_locate 里才清洗。
        # 这里直接用原始值做 Key 也没问题，只要原始值一样，结果就该一样。
        cache_key = (city_key, addr_key)
        
        # 2. 查缓存
        if cache_key in addr_cache:
            # 【命中缓存！】直接使用上次的结果
            lng, lat, strategy, match_addr, used_addr, level, match_count, sim_score = addr_cache[cache_key]
            # 为了区分，可以在策略里标记一下（可选）
            # strategy += "_Cached" 
            cache_hits += 1
        else:
            # 【未命中】调用 API
            result_tuple = smart_locate(row)
            
            # 解包检查是否是限额报错
            if result_tuple[0] == "LIMIT_REACHED":
                print("\n[程序终止] API 额度耗尽。")
                save_batch(batch_buffer, OUTPUT_FILE)
                break
                
            lng, lat, strategy, match_addr, used_addr, level, match_count, sim_score = result_tuple
            
            # 只有当成功找到时，才存入缓存 (避免缓存 Not_Found，万一只是网络抖动呢)
            # 或者你也可以缓存 Not_Found，看你需求。这里建议只缓存成功的。
            if lng is not None:
                addr_cache[cache_key] = result_tuple
        
        # 3. 写入结果
        res = row.copy()
        res['clean_search_addr'] = used_addr 
        res['longitude'] = lng
        res['latitude'] = lat
        res['locate_strategy'] = strategy   
        res['amap_matched'] = match_addr    
        res['match_level'] = level 
        res['match_count'] = match_count
        res['similarity'] = round(sim_score, 4)
        
        batch_buffer.append(res)
        
        if len(batch_buffer) >= BATCH_SIZE:
            save_batch(batch_buffer, OUTPUT_FILE)
            batch_buffer = []
            
        if (count + 1) % 50 == 0:
            print(f"\r进度:{count+1}/{total_rows} | 缓存命中:{cache_hits} | 最新:{match_addr[:10]}...", end="")
            
        count += 1
        # 缓存命中时不需要 sleep，大大加速
        if cache_key not in addr_cache: 
            time.sleep(0.4) 

    if batch_buffer:
        save_batch(batch_buffer, OUTPUT_FILE)
    print("\n完成。")

if __name__ == "__main__":
    main()

正在读取输入文件: D:\SM\zdxrun\datatest\llm结果\all city\real_missing_tasks_final.csv ...
已加载本地缓存: 127750 条
剩余任务: 4520 条
进度:4000/4520 | 缓存命中:2584 | 最新:浙江省丽水市莲都区塔...
完成。


In [6]:
import pandas as pd

# 读取结果文件
df = pd.read_csv(r'D:\SM\zdxrun\datatest\llm结果\all city\final_coords_result.csv', dtype=str)

# 转换数值列
df['match_count'] = pd.to_numeric(df['match_count'], errors='coerce').fillna(0)
df['similarity'] = pd.to_numeric(df['similarity'], errors='coerce').fillna(0)

print("===== 1. 匹配层级 (Match Level) 分布 =====")
# 这是判断精度的最核心指标
level_counts = df['match_level'].value_counts()
print(level_counts)
print(f"\n总行数: {len(df)}")

print("\n===== 2. 策略与层级交叉分析 =====")
# 看看HighQuality策略下，到底有多少是真正的高质量
print(pd.crosstab(df['locate_strategy'], df['match_level']))

print("\n===== 3. 多结果统计 (Match Count > 1) =====")
multi_match = df[df['match_count'] > 1]
print(f"返回多个坐标的地址数量: {len(multi_match)} (占比 {len(multi_match)/len(df):.2%})")
print("多结果数据的层级分布：")
print(multi_match['match_level'].value_counts().head(5))

print("\n===== 4. 相似度分布概览 =====")
print(df['similarity'].describe())

===== 1. 匹配层级 (Match Level) 分布 =====
match_level
道路        63341
市         48022
住宅区       30696
区县        26712
兴趣点       26083
村庄        24334
乡镇        18053
公交地铁站点    13372
道路交叉路口     5461
未知         4356
省          3040
POI        1345
门址         1081
门牌号         290
小巷          123
Name: count, dtype: int64

总行数: 266659

===== 2. 策略与层级交叉分析 =====
match_level                 POI     乡镇    住宅区  公交地铁站点    兴趣点     区县   小巷  \
locate_strategy                                                            
GeoCode_HighQuality           0  17843  30526   13322  25818  26647  123   
GeoCode_HighQuality_LowSim    0      5      2      10     62      0    0   
POI_Corrected                79      0      0       0      0      0    0   
POI_Corrected_LowSim          8      0      0       0      0      0    0   
POI_Search                  988      0      0       0      0      0    0   
POI_Search_LowSim           270      0      0       0      0      0    0   
WRONG_CITY_DRIFT              0    204

In [14]:
import pandas as pd
import os

# ================= 配置区域 =================
INPUT_FILE = r'D:\SM\zdxrun\datatest\llm结果\all city\final_coords_result2.csv'
OUTPUT_DIR = r'D:\SM\zdxrun\datatest\llm结果\all city'

# 定义筛选标准
# 1. 我们要保留的所有层级（High + Medium）
VALID_LEVELS = [
    # --- 高精度 ---
    '门址', '门牌号', 'POI', '兴趣点', '住宅区', '村庄', '热点', '小巷', 
    # --- 中精度 (道路/乡镇) ---
    '道路', '道路交叉路口', '公交地铁站点'
]

# 2. 明确要丢弃的垃圾层级 (用于核对)
TRASH_LEVELS = [
    '省', '市', '区县', '未知', 'None', '', '乡镇'
]

# ================= 主逻辑 =================
def merge_and_filter():
    print("正在读取数据...")
    if not os.path.exists(INPUT_FILE):
        print("文件不存在！")
        return

    df = pd.read_csv(INPUT_FILE, dtype=str)
    
    # 简单的预处理
    df['match_count'] = pd.to_numeric(df['match_count'], errors='coerce').fillna(0)
    df['similarity'] = pd.to_numeric(df['similarity'], errors='coerce').fillna(0)
    df['match_level'] = df['match_level'].fillna('未知')

    print(f"原始数据总行数: {len(df)}")

    # ---------------- 核心筛选逻辑 ----------------
    # 条件1: match_level 在我们在定义的有效列表里
    # 条件2: similarity 不能太低 (比如 < 0.4 通常是完全错误的匹配)
    # 注意：对于“道路”级，相似度有时候会偏低，所以只要不是特别低(0.4)都可以留
    
    mask_keep = (
        df['match_level'].isin(VALID_LEVELS) & 
        (df['similarity'] >= 0.4) 
    )

    # 分割数据
    df_valid = df[mask_keep].copy()
    df_trash = df[~mask_keep].copy()

    # ---------------- 统计与输出 ----------------
    print("\n===== 筛选结果 =====")
    print(f"✅ 有效数据 (合并了精确点+道路): {len(df_valid)} 条")
    print(f"   占比: {len(df_valid)/len(df):.2%}")
    print(f"❌ 废弃数据 (省/市/区县/低相似度): {len(df_trash)} 条")
    
    print("\n有效数据中的层级分布前5名：")
    print(df_valid['match_level'].value_counts().head(5))

    # 保存
    valid_path = os.path.join(OUTPUT_DIR, 'cleaned_valid_data.csv')
    trash_path = os.path.join(OUTPUT_DIR, 'cleaned_trash_data.csv')

    df_valid.to_csv(valid_path, index=False, encoding='utf-8-sig')
    df_trash.to_csv(trash_path, index=False, encoding='utf-8-sig')
    
    print(f"\n文件已保存至:\n1. {valid_path} (主要使用这个)\n2. {trash_path}")

if __name__ == "__main__":
    merge_and_filter()

正在读取数据...
原始数据总行数: 271179

===== 筛选结果 =====
✅ 有效数据 (合并了精确点+道路): 164379 条
   占比: 60.62%
❌ 废弃数据 (省/市/区县/低相似度): 106800 条

有效数据中的层级分布前5名：
match_level
道路        63604
住宅区       31058
兴趣点       25080
村庄        24396
公交地铁站点    12922
Name: count, dtype: int64

文件已保存至:
1. D:\SM\zdxrun\datatest\llm结果\all city\cleaned_valid_data.csv (主要使用这个)
2. D:\SM\zdxrun\datatest\llm结果\all city\cleaned_trash_data.csv


# 时间和id修复

In [7]:
import pandas as pd

# 1. 定义文件路径
source_file = r'D:\SM\zdxrun\datatest\event_weibo_new\all_flood_weibos_12years_fixed.csv'
target_file = r'D:\SM\zdxrun\datatest\llm结果\all city\int0212_patched_wgs84.csv'
output_file = r'D:\SM\zdxrun\datatest\llm结果\all city\int0212_patched_wgs84_repaired.csv'

try:
    print("正在读取完美的源文件 (词典)...")
    # dtype=str 极其重要！防止微博ID被变成科学计数法
    df_source = pd.read_csv(source_file, encoding='utf-8-sig', dtype=str)
    
    print("正在读取需要修复的目标文件...")
    df_target = pd.read_csv(target_file, encoding='utf-8-sig', dtype=str)
    
    total_target_rows = len(df_target)
    print(f"目标文件共有数据: {total_target_rows} 条")
    
    # 2. 制作匹配的“钥匙”（清理换行符和首尾空格，提高匹配成功率）
    print("正在建立匹配索引...")
    # 对源文件正文进行清理，并去重（保留第一个出现的），制作成纯净的参照表
    df_source['match_key'] = df_source['微博正文'].astype(str).str.strip().str.replace('\r', '').str.replace('\n', '')
    lookup_table = df_source.drop_duplicates(subset=['match_key'])[['match_key', '微博ID', '发布时间']]
    # 重命名列名，防止与目标文件冲突
    lookup_table = lookup_table.rename(columns={'微博ID': '正确ID', '发布时间': '正确时间'})
    
    # 对目标文件正文也进行同样的清理，生成匹配钥匙
    df_target['match_key'] = df_target['微博正文'].astype(str).str.strip().str.replace('\r', '').str.replace('\n', '')
    
    # 3. 执行匹配 (Left Join 类似 VLOOKUP)
    print("正在通过微博正文执行匹配修复...")
    df_merged = pd.merge(df_target, lookup_table, on='match_key', how='left')
    
    # 4. 统计与修复
    # 找出匹配成功的行（即在源文件中找到了对应的正文）
    success_mask = df_merged['正确ID'].notna()
    success_count = success_mask.sum()
    failed_count = total_target_rows - success_count
    
    # 将正确的 ID 和 时间 覆盖回目标文件的对应列
    df_merged.loc[success_mask, '微博ID'] = df_merged.loc[success_mask, '正确ID']
    df_merged.loc[success_mask, '发布时间'] = df_merged.loc[success_mask, '正确时间']
    
    # 5. 清理临时生成的辅助列
    df_final = df_merged.drop(columns=['match_key', '正确ID', '正确时间'])
    
    # 6. 打印统计报告
    print("\n" + "="*30)
    print("         修复统计报告")
    print("="*30)
    print(f"目标文件总行数: {total_target_rows} 条")
    print(f"✅ 成功匹配并修复: {success_count} 条 (占比 {success_count/total_target_rows:.2%})")
    
    if failed_count > 0:
        print(f"❌ 未能找到匹配项: {failed_count} 条")
        print("   (原因可能是：LLM大模型修改了正文内容，或者正文为空)")
    else:
        print("🎉 完美！所有数据均已100%匹配并修复！")
    print("="*30 + "\n")
    
    # 7. 保存修复后的文件
    print("正在保存修复后的文件...")
    df_final.to_csv(output_file, index=False, encoding='utf-8-sig')
    print(f"大功告成！文件已保存至：\n{output_file}")

except Exception as e:
    print(f"处理时发生错误: {e}")
    print("如果是编码报错，请尝试把 encoding='utf-8-sig' 改为 encoding='gbk' 或 'gb18030'")

正在读取完美的源文件 (词典)...
正在读取需要修复的目标文件...
目标文件共有数据: 156550 条
正在建立匹配索引...
正在通过微博正文执行匹配修复...

         修复统计报告
目标文件总行数: 156550 条
✅ 成功匹配并修复: 156546 条 (占比 100.00%)
❌ 未能找到匹配项: 4 条
   (原因可能是：LLM大模型修改了正文内容，或者正文为空)

正在保存修复后的文件...
大功告成！文件已保存至：
D:\SM\zdxrun\datatest\llm结果\all city\int0212_patched_wgs84_repaired.csv


# llm精度检验

In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd
import os
import numpy as np
import warnings
from Levenshtein import distance as levenshtein_distance
from rouge import Rouge
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score, confusion_matrix
from bert_score import BERTScorer # <--- 新增
from prettytable import PrettyTable # <--- 新增用于美化表格

# 忽略一些警告
warnings.filterwarnings("ignore")

# ================= 配置区域 =================
# 输入文件
INPUT_FILE = r'D:\SM\zdxrun\datatest\llm结果\48shi\benchmark\gold_standard_vllm_updated5_analyzed.csv'

# 输出文件：错题本
OUTPUT_ERROR_FILE = r'D:\SM\zdxrun\datatest\llm结果\48shi\benchmark\error_analysis_scientific.csv'

# ROUGE 计算器初始化
rouge_scorer = Rouge()
# ===========================================

def parse_bool(val):
    """清洗布尔值"""
    return str(val).strip().lower() in ['true', '1', 'yes', 't', 'ture']

def get_clean_set(val):
    """将单元格文本转为集合 (去除空值和重复)"""
    if pd.isna(val): return []
    items = [s.strip() for s in str(val).split('\n') if s.strip()]
    return sorted(list(set(items)))

def calculate_rouge_l_score(pred_list, gt_list):
    """计算 ROUGE-L F1 分数"""
    if not pred_list and not gt_list: return 1.0
    if not pred_list or not gt_list: return 0.0
    
    pred_str = " ".join(pred_list)
    gt_str = " ".join(gt_list)
    
    try:
        scores = rouge_scorer.get_scores(pred_str, gt_str)
        return scores[0]['rouge-l']['f']
    except:
        return 0.0

def calculate_char_similarity(pred_list, gt_list):
    """计算基于编辑距离的字符级相似度 (Levenshtein)"""
    if not pred_list and not gt_list: return 1.0
    if not pred_list or not gt_list: return 0.0
    
    pred_str = "".join(pred_list)
    gt_str = "".join(gt_list)
    
    dist = levenshtein_distance(pred_str, gt_str)
    max_len = max(len(pred_str), len(gt_str))
    
    if max_len == 0: return 1.0
    return 1.0 - (dist / max_len)

def calculate_bert_score(scorer, pred_list, gt_list):
    """
    计算 BERTScore (语义相似度 F1)
    """
    if not pred_list and not gt_list: return 1.0
    if not pred_list or not gt_list: return 0.0

    # BERTScore 通常对比句子，这里我们将列表合并为字符串进行语义对比
    # 中文建议用逗号或空格连接
    pred_str = "，".join(pred_list)
    gt_str = "，".join(gt_list)

    try:
        # scorer.score 返回 (P, R, F1)，我们需要 F1
        # 输入必须是 list 形式
        P, R, F1 = scorer.score([pred_str], [gt_str])
        return F1.item() # 转为 Python float
    except Exception as e:
        print(f"BERTScore计算出错: {e}")
        return 0.0

def evaluate_scientific_conditional(df, llm_col, gt_col, field_name, error_list, bert_scorer):
    """
    【全能评估函数】包含 BERTScore
    """
    scores_rouge = []
    scores_char = []
    scores_bert = [] # <--- 存储 BERT 分数
    
    soft_tp, soft_fp, soft_fn = 0, 0, 0
    valid_count = 0
    
    # 打印进度提示，因为 BERT 计算较慢
    print(f"正在评估字段: [{field_name}] (包含语义分析)...")

    for _, row in df.iterrows():
        # 1. 条件过滤：只看双 True 样本
        is_gt_w = parse_bool(row['GT_is_w'])
        is_llm_w = parse_bool(row['llm_is_w'])
        
        if not (is_gt_w and is_llm_w):
            continue 
            
        valid_count += 1
        
        # 2. 准备数据
        llm_list = get_clean_set(row[llm_col])
        gt_list = get_clean_set(row[gt_col])
        
        # --- 指标 A: ROUGE-L ---
        r_score = calculate_rouge_l_score(llm_list, gt_list)
        scores_rouge.append(r_score)
        
        # --- 指标 B: Character Similarity ---
        c_score = calculate_char_similarity(llm_list, gt_list)
        scores_char.append(c_score)

        # --- 指标 C: BERT Score (新增) ---
        b_score = calculate_bert_score(bert_scorer, llm_list, gt_list)
        scores_bert.append(b_score)
        
        # --- 指标 D: Entity Soft-Match F1 ---
        temp_gt = gt_list.copy()
        matched_indices = set()
        
        current_tp = 0
        for i, pred in enumerate(llm_list):
            for j, gt in enumerate(temp_gt):
                # 宽容逻辑：A包含B 或 B包含A
                if pred in gt or gt in pred:
                    current_tp += 1
                    temp_gt.pop(j)
                    matched_indices.add(i)
                    break
        
        soft_tp += current_tp
        soft_fp += (len(llm_list) - len(matched_indices))
        soft_fn += len(temp_gt)
        
        # 3. 收集错题 (增加了 BERT 分数记录)
        # 如果完全匹配失败，或者分数很低，都算错题
        is_bad_case = (len(temp_gt) > 0 or (len(llm_list) - len(matched_indices)) > 0)
        
        if is_bad_case:
            error_list.append({
                'id': row.get('id', ''),
                'text': row.get('text', ''),
                'Field': field_name,
                'GT': str(gt_list),
                'LLM': str(llm_list),
                'Soft-F1': 'Mismatch',
                'ROUGE-L': round(r_score, 4),
                'Char-Sim': round(c_score, 4),
                'BERT-Sim': round(b_score, 4)
            })

    # === 汇总结果 ===
    avg_rouge = np.mean(scores_rouge) if scores_rouge else 0
    avg_char = np.mean(scores_char) if scores_char else 0
    avg_bert = np.mean(scores_bert) if scores_bert else 0
    
    p = soft_tp / (soft_tp + soft_fp + 1e-9)
    r = soft_tp / (soft_tp + soft_fn + 1e-9)
    f1_soft = 2 * p * r / (p + r + 1e-9)
    
    return {
        'Field': field_name,
        'Samples': valid_count,
        'Soft-F1': f1_soft,
        'Precision': p,
        'Recall': r,
        'ROUGE-L': avg_rouge,
        'Char-Sim': avg_char,
        'BERT-Sim': avg_bert
    }

def print_pretty_classification(acc, prec, rec, f1, tn, fp, fn, tp):
    """打印分类任务表格"""
    print("\n" + "="*30 + " 任务一：事件判定 (Binary Classification) " + "="*30)
    
    # 1. 指标表
    table = PrettyTable()
    table.field_names = ["Metric", "Value", "Description"]
    table.align["Metric"] = "l"
    table.align["Description"] = "l"
    
    table.add_row(["Accuracy", f"{acc:.4f}", "总体准确率"])
    table.add_row(["Precision", f"{prec:.4f}", "查准率 (预测为True中有多少是对的)"])
    table.add_row(["Recall", f"{rec:.4f}", "查全率 (真实为True中找出了多少)"])
    table.add_row(["F1 Score", f"{f1:.4f}", "综合指标"])
    print(table)

    # 2. 混淆矩阵表
    cm_table = PrettyTable()
    cm_table.field_names = ["Confusion Matrix", "Predicted: NO", "Predicted: YES"]
    cm_table.add_row(["Actual: NO", f"TN: {tn}", f"FP: {fp} (误报)"])
    cm_table.add_row(["Actual: YES", f"FN: {fn} (漏报)", f"TP: {tp}"])
    print(cm_table)

def print_pretty_extraction(results):
    """打印提取任务表格"""
    print("\n" + "="*30 + " 任务二：信息提取 (Information Extraction) " + "="*30)
    
    table = PrettyTable()
    # 定义表头
    table.field_names = [
        "Field", "Soft-F1", "Precision", "Recall", 
        "ROUGE-L", "Char-Sim", "BERT-Sim"
    ]
    # 设置对齐
    table.align = "r"
    table.align["Field"] = "l"
    
    for res in results:
        table.add_row([
            res['Field'],
            f"{res['Soft-F1']:.4f}",
            f"{res['Precision']:.4f}",
            f"{res['Recall']:.4f}",
            f"{res['ROUGE-L']:.4f}",
            f"{res['Char-Sim']:.4f}",
            f"{res['BERT-Sim']:.4f}" # <--- 显示 BERT 分数
        ])
    
    print(table)
    print("\n*注: Soft-F1为实体级模糊匹配; BERT-Sim为语义相似度; Char-Sim为字面编辑距离。")

def main():
    if not os.path.exists(INPUT_FILE):
        print(f"找不到文件: {INPUT_FILE}")
        return

    print(f"正在读取数据: {os.path.basename(INPUT_FILE)}...")
    try:
        df = pd.read_csv(INPUT_FILE, dtype=str).fillna('')
    except:
        df = pd.read_csv(INPUT_FILE, dtype=str, encoding='gb18030').fillna('')
    
    # === 初始化 BERT Scorer ===
    print("正在加载 BERT 模型 (初次运行需下载模型, 且比较耗时)...")
    # lang="zh" 会自动下载 bert-base-chinese 
    # rescale_with_baseline=True 通常建议开启以获得更好看的分布，但需要额外文件，这里简化设为False
    try:
        bert_scorer = BERTScorer(lang="zh", rescale_with_baseline=False)
        print("BERT 模型加载完成。")
    except Exception as e:
        print(f"BERT 模型加载失败，请检查网络或是否安装pytorch: {e}")
        bert_scorer = None

    all_bad_cases = []
    extraction_results = []

    # ========================================================
    # 1. 分类任务评估 (is_w)
    # ========================================================
    y_true = df['GT_is_w'].apply(parse_bool)
    y_pred = df['llm_is_w'].apply(parse_bool)
    
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    
    try:
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    except:
        tn, fp, fn, tp = 0, 0, 0, 0

    # 打印分类结果
    print_pretty_classification(acc, prec, rec, f1, tn, fp, fn, tp)

    # ========================================================
    # 2. 深度评估提取项 (带 BERT)
    # ========================================================
    extract_targets = [
        ('llm_city', 'GT_city', 'City'),
        ('llm_loc',  'GT_loc',  'Location'),
        ('llm_stat', 'GT_stat', 'Status'),
        ('llm_time', 'GT_time', 'Time')
    ]
    
    for llm_c, gt_c, name in extract_targets:
        if llm_c in df.columns:
            if bert_scorer:
                res = evaluate_scientific_conditional(df, llm_c, gt_c, name, all_bad_cases, bert_scorer)
                extraction_results.append(res)
            else:
                print("跳过详细评估，因为模型未加载。")

    # 打印提取结果
    print_pretty_extraction(extraction_results)

    # ========================================================
    # 3. 自动生成结论建议
    # ========================================================
    print("\n" + "="*60)
    print("【论文摘要/结论生成助手】")
    if extraction_results:
        # 计算平均 BERT 分数
        avg_bert = np.mean([r['BERT-Sim'] for r in extraction_results])
        print(f"Overall Semantic Similarity (BERTScore): {avg_bert:.4f}")
        
        if avg_bert > 0.85:
            print(">> 结论: 模型提取的内容在语义上与真值高度一致。")
        elif avg_bert > 0.7:
            print(">> 结论: 模型提取内容语义相关，但可能存在措辞差异。")
        else:
            print(">> 结论: 模型可能存在明显的语义偏差或幻觉。")

    # 4. 保存错题
    if all_bad_cases:
        pd.DataFrame(all_bad_cases).to_csv(OUTPUT_ERROR_FILE, index=False, encoding='utf-8-sig')
        print(f"\n[Done] 错题本已保存至: {OUTPUT_ERROR_FILE}")

if __name__ == "__main__":
    main()